In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/tanmayistired/notebookc1f8955f15/__results__.html
/kaggle/input/notebooks/tanmayistired/notebookc1f8955f15/__notebook__.ipynb
/kaggle/input/notebooks/tanmayistired/notebookc1f8955f15/__output__.json
/kaggle/input/notebooks/tanmayistired/notebookc1f8955f15/custom.css
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/README_KAGGLE_FINAL.md
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/KAGGLE_BOOTSTRAP.py
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/OLD_AMLC2026_RESUME_BOOTSTRAP.py
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/README_KAGGLE.md
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/OLD_artifact_manifest.json
/kaggle/

In [2]:
# ============================================================
# AMLC 2026 — KAGGLE CELL 1
# EXACT CHECKPOINT RESTORE + ENVIRONMENT VERIFICATION
# ============================================================

import os
import gc
import json
import psutil
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq


# ============================================================
# 1) EXACT PACKAGE ROOT
# ============================================================

WORKSPACE_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/"
    "amlc-2026-final-workspace/"
    "AMLC2026_KAGGLE_FINAL"
)

CHECKPOINT_ROOT = (
    WORKSPACE_ROOT /
    "checkpoint_after_cell58_FINAL_20260925"
)

DATASET_ROOT = (
    WORKSPACE_ROOT /
    "dataset"
)

TRAIN_ROOT = DATASET_ROOT / "train"
TEST_ROOT = DATASET_ROOT / "test"

STATE_ROOT = CHECKPOINT_ROOT / "state"
RUNTIME_ROOT = CHECKPOINT_ROOT / "runtime"
BLOCKING_ROOT = CHECKPOINT_ROOT / "blocking"
FULL_TRAIN_ROOT = CHECKPOINT_ROOT / "full_training"


# ============================================================
# 2) BASIC EXISTENCE CHECK
# ============================================================

print("=" * 72)
print("AMLC 2026 — KAGGLE CHECKPOINT RESTORE")
print("=" * 72)

required_dirs = {
    "WORKSPACE_ROOT": WORKSPACE_ROOT,
    "CHECKPOINT_ROOT": CHECKPOINT_ROOT,
    "DATASET_ROOT": DATASET_ROOT,
    "TRAIN_ROOT": TRAIN_ROOT,
    "TEST_ROOT": TEST_ROOT,
    "STATE_ROOT": STATE_ROOT,
    "RUNTIME_ROOT": RUNTIME_ROOT,
}

for name, path in required_dirs.items():
    print(
        f"{'OK   ' if path.exists() else 'MISS '}{name}: {path}"
    )

missing_dirs = [
    name
    for name, path in required_dirs.items()
    if not path.exists()
]

if missing_dirs:
    raise RuntimeError(
        "Missing required directories:\n" +
        "\n".join(missing_dirs)
    )


# ============================================================
# 3) KEY ARTIFACT PATHS
# ============================================================

CANDIDATE_PATH = (
    STATE_ROOT /
    "eval_candidates_address_rescue.parquet"
)

CANDIDATE_COUNTS_PATH = (
    STATE_ROOT /
    "eval_candidate_counts_address_rescue.parquet"
)

CANDIDATE_RESIDUAL_PATH = (
    STATE_ROOT /
    "eval_candidate_residual_address_rescue.parquet"
)

S1_LOOKUP_PATH = (
    RUNTIME_ROOT /
    "s1_pair_lookup.parquet"
)

S2_NAME_PATH = (
    RUNTIME_ROOT /
    "s2_name_lookup.parquet"
)

S2_ADDRESS_PATH = (
    RUNTIME_ROOT /
    "s2_address_lookup.parquet"
)

S3_NAME_PATH = (
    RUNTIME_ROOT /
    "s3_name_lookup.parquet"
)

S3_ADDRESS_PATH = (
    RUNTIME_ROOT /
    "s3_address_lookup.parquet"
)

GT_PATH = (
    FULL_TRAIN_ROOT /
    "gt.parquet"
)

GT_EDGES_PATH = (
    FULL_TRAIN_ROOT /
    "gt_edges.parquet"
)

CHECKPOINT_MANIFEST = (
    CHECKPOINT_ROOT /
    "CHECKPOINT_MANIFEST.json"
)


# ============================================================
# 4) VERIFY CRITICAL FILES
# ============================================================

critical_files = {
    "candidate_pairs": CANDIDATE_PATH,
    "candidate_counts": CANDIDATE_COUNTS_PATH,
    "candidate_residual": CANDIDATE_RESIDUAL_PATH,
    "s1_lookup": S1_LOOKUP_PATH,
    "s2_name_lookup": S2_NAME_PATH,
    "s2_address_lookup": S2_ADDRESS_PATH,
    "s3_name_lookup": S3_NAME_PATH,
    "s3_address_lookup": S3_ADDRESS_PATH,
    "gt": GT_PATH,
    "gt_edges": GT_EDGES_PATH,
    "manifest": CHECKPOINT_MANIFEST,
}

print("\nCritical artifacts:")

for name, path in critical_files.items():
    print(
        f"{'OK   ' if path.exists() else 'MISS '}{name}: {path.name}"
    )

missing_files = [
    name
    for name, path in critical_files.items()
    if not path.exists()
]

if missing_files:
    raise RuntimeError(
        "Missing critical checkpoint files:\n" +
        "\n".join(missing_files)
    )


# ============================================================
# 5) READ MANIFEST
# ============================================================

with open(
    CHECKPOINT_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    CHECKPOINT_INFO = json.load(f)

print("\nCheckpoint status:")
print(
    "  status:",
    CHECKPOINT_INFO.get("status")
)

print(
    "  stopping_point:",
    CHECKPOINT_INFO.get("stopping_point")
)

print(
    "  candidate_pairs:",
    f"{CHECKPOINT_INFO.get('candidate_pairs', 0):,}"
)


# ============================================================
# 6) VERIFY CANDIDATE PARQUET WITHOUT LOADING IT
# ============================================================

candidate_meta = pq.ParquetFile(
    str(CANDIDATE_PATH)
).metadata

candidate_rows = candidate_meta.num_rows

print("\nCandidate artifact:")
print(
    "  rows:",
    f"{candidate_rows:,}"
)

print(
    "  size:",
    round(
        CANDIDATE_PATH.stat().st_size /
        1024**2,
        2
    ),
    "MB"
)

assert candidate_rows == 22_302_012, (
    f"Unexpected candidate count: {candidate_rows:,}"
)


# ============================================================
# 7) READ ONLY SMALL / MANAGEABLE EVAL OBJECTS
# ============================================================

def read_parquet(path):
    return pl.read_parquet(path)


s1_eval = read_parquet(
    RUNTIME_ROOT /
    "s1_eval.parquet"
)

edge_eval = read_parquet(
    RUNTIME_ROOT /
    "edge_eval.parquet"
)

eval_gt = read_parquet(
    RUNTIME_ROOT /
    "eval_gt.parquet"
)

per_s1_recall = read_parquet(
    RUNTIME_ROOT /
    "per_s1_recall.parquet"
)

positive_pair_diagnostics = read_parquet(
    RUNTIME_ROOT /
    "positive_pair_diagnostics.parquet"
)

print("\nEvaluation state:")
print(
    "  s1_eval:",
    f"{s1_eval.height:,} rows"
)

print(
    "  edge_eval:",
    f"{edge_eval.height:,} rows"
)

print(
    "  eval_gt:",
    f"{eval_gt.height:,} rows"
)

print(
    "  per_s1_recall:",
    f"{per_s1_recall.height:,} rows"
)


# ============================================================
# 8) LOAD S1 LOOKUP
#
# Only 100k rows — safe.
# ============================================================

s1_lookup = pl.read_parquet(
    S1_LOOKUP_PATH
)

print(
    "\nS1 lookup:",
    s1_lookup.shape
)


# ============================================================
# 9) KEEP LARGE TABLES ON DISK
#
# These are PATHS, NOT in-memory DataFrames.
# Cell 59 will stream/join them in controlled chunks.
# ============================================================

S2_NAME_PATH = Path(S2_NAME_PATH)
S2_ADDRESS_PATH = Path(S2_ADDRESS_PATH)
S3_NAME_PATH = Path(S3_NAME_PATH)
S3_ADDRESS_PATH = Path(S3_ADDRESS_PATH)


# ============================================================
# 10) VERIFY RAW COMPETITION DATA
# ============================================================

TRAIN_FILES = sorted(
    p.name
    for p in TRAIN_ROOT.iterdir()
    if p.is_file()
)

TEST_FILES = sorted(
    p.name
    for p in TEST_ROOT.iterdir()
    if p.is_file()
)

print("\nTrain files:")
for x in TRAIN_FILES:
    print(" ", x)

print("\nTest files:")
for x in TEST_FILES:
    print(" ", x)

expected_train = {
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
}

expected_test = {
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv",
}

assert expected_train.issubset(
    set(TRAIN_FILES)
), "Train dataset files missing."

assert expected_test.issubset(
    set(TEST_FILES)
), "Test dataset files missing."


# ============================================================
# 11) FINAL RESOURCE REPORT
# ============================================================

mem = psutil.virtual_memory()

print("\n" + "=" * 72)
print("KAGGLE RESTORE SUCCESSFUL")
print("=" * 72)

print(
    "\nRAM:",
    round(mem.used / 1024**3, 2),
    "GB /",
    round(mem.total / 1024**3, 2),
    "GB"
)

print(
    "\nCandidate pairs:",
    f"{candidate_rows:,}"
)

print(
    "\nS1 eval:",
    f"{s1_eval.height:,}"
)

print(
    "\nEverything required for Cell 59 is present."
)

print(
    "\nIMPORTANT:"
)

print(
    "The 22.3M candidates are NOT loaded into RAM."
)

print(
    "The 5M+ S2/S3 lookup tables are NOT loaded into RAM."
)

print(
    "They will be streamed/queried during Cell 59."
)

print(
    "\nNEXT → KAGGLE CELL 2 = PAIRWISE BASELINE / CELL 59"
)

print("=" * 72)

AMLC 2026 — KAGGLE CHECKPOINT RESTORE
OK   WORKSPACE_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL
OK   CHECKPOINT_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925
OK   DATASET_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset
OK   TRAIN_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train
OK   TEST_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/test
OK   STATE_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/state
OK   RUNTIME_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/runtime

Critical artifacts:
OK   candidate_pairs: eval_candidates_address_res

In [3]:
# ============================================================
# AMLC 2026 — KAGGLE DEPENDENCY BOOTSTRAP
# Run BEFORE CELL 59
# ============================================================

import sys
import subprocess
import importlib.util

REQUIRED = {
    "rapidfuzz": "rapidfuzz==3.14.6",
    "lightgbm": "lightgbm==4.6.0",
    "duckdb": "duckdb==1.3.2",
    "polars": "polars==1.35.2",
    "pyarrow": "pyarrow",
    "joblib": "joblib",
    "psutil": "psutil",
}

missing = []

for module, package in REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        missing.append(package)

print("Missing packages:")
for x in missing:
    print("  ", x)

if missing:
    print("\nInstalling...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing,
    ])
else:
    print("\nAll required packages already installed.")

print("\nVerification:")

for module in REQUIRED:
    try:
        mod = __import__(module)
        version = getattr(mod, "__version__", "installed")
        print(f"  OK   {module}: {version}")
    except Exception as e:
        print(f"  FAIL {module}: {e}")

Missing packages:
   rapidfuzz==3.14.6

Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.7 MB/s eta 0:00:00

Verification:
  OK   rapidfuzz: 3.14.6
  OK   lightgbm: 4.6.0
  OK   duckdb: 1.3.2
  OK   polars: 1.35.2
  OK   pyarrow: 24.0.0
  OK   joblib: 1.5.3
  OK   psutil: 5.9.5


In [4]:
# ============================================================
# REPAIR — restore candidate_recall_ceiling
# ============================================================

candidate_recall_ceiling = {
    "candidate_pairs": 22_302_012,
    "edge_recall_overall": 229_692 / 345_980,
    "edge_recall_s2": 0.673978,
    "edge_recall_s3": 0.654444,
    "full_set_recovery": 33_746 / 94_404,
    "eval_s1": 100_000,
    "eval_positive_edges": 345_980,
    "eval_full_positive_sets": 94_404,
}

print("candidate_recall_ceiling restored:")
for k, v in candidate_recall_ceiling.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6%}" if v <= 1 else f"  {k}: {v:,}")
    else:
        print(f"  {k}: {v:,}" if isinstance(v, int) else f"  {k}: {v}")

print("\n✅ Ready to rerun CELL 59 continuation.")

candidate_recall_ceiling restored:
  candidate_pairs: 22,302,012
  edge_recall_overall: 66.388809%
  edge_recall_s2: 67.397800%
  edge_recall_s3: 65.444400%
  full_set_recovery: 35.746367%
  eval_s1: 100,000
  eval_positive_edges: 345,980
  eval_full_positive_sets: 94,404

✅ Ready to rerun CELL 59 continuation.


In [5]:
# ==============================================================================
# AMLC 2026 — CELL 59 — SELF-CONTAINED PAIR-SAMPLE BRIDGE
#
# PURPOSE:
#   Recover enough state to continue into feature engineering.
#
# IMPORTANT:
#   This intentionally DOES NOT validate source == {"source2","source3"}.
#   The surviving Cell-58 parquet uses a different source encoding.
#
#   We preserve the original `source` column exactly as stored and derive
#   `source_is_s3` robustly.
# ==============================================================================

from pathlib import Path
import json
import time
import numpy as np
import polars as pl

print("=" * 78)
print("AMLC 2026 — CELL 59 — SELF-CONTAINED PAIR-SAMPLE BRIDGE")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------

MASTER_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL"
)

CANDIDATE_PATH = (
    MASTER_ROOT
    / "checkpoint_after_cell58_FINAL_20260925"
    / "state"
    / "eval_candidates_tier12_source_capped.parquet"
)

GT_PATH = (
    MASTER_ROOT
    / "dataset"
    / "train"
    / "train_ground_truth.tsv"
)

PAIR_SAMPLE_ROOT = Path(
    "/kaggle/working/AMLC2026/pair_sample_cell58"
)
PAIR_SAMPLE_ROOT.mkdir(parents=True, exist_ok=True)

BASELINE_PAIR_PATH = (
    PAIR_SAMPLE_ROOT / "baseline_candidate_pairs.parquet"
)

METADATA_PATH = (
    PAIR_SAMPLE_ROOT / "cell59_bridge_metadata.json"
)

TARGET_ROWS = 3_429_214
SEED = 2026

print(f"\nCandidate : {CANDIDATE_PATH}")
print(f"GT        : {GT_PATH}")
print(f"Output    : {BASELINE_PAIR_PATH}")

assert CANDIDATE_PATH.exists(), f"Candidate artifact missing: {CANDIDATE_PATH}"
assert GT_PATH.exists(), f"Ground truth missing: {GT_PATH}"

# ------------------------------------------------------------------------------
# 1. LOAD CANDIDATE POOL
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. LOAD SURVIVING CELL-58 CANDIDATE POOL")
print("-" * 78)

cand = pl.read_parquet(CANDIDATE_PATH)

print(f"Rows: {cand.height:,}")
print(f"Columns: {cand.columns}")

required = {
    "s1_entity_id",
    "candidate_entity_id",
    "source",
}

missing = required - set(cand.columns)

assert not missing, f"Missing columns: {sorted(missing)}"

cand = cand.with_columns([
    pl.col("s1_entity_id").cast(pl.Utf8),
    pl.col("candidate_entity_id").cast(pl.Utf8),
    pl.col("source").cast(pl.Utf8),
])

# ------------------------------------------------------------------------------
# 2. INSPECT ACTUAL SOURCE ENCODING — NO ASSERTION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. ACTUAL SOURCE VALUES")
print("-" * 78)

source_values = (
    cand
    .select("source")
    .unique()
    .sort("source")
)

print(source_values)

source_counts = (
    cand
    .group_by("source")
    .agg(pl.len().alias("rows"))
    .sort("source")
)

print("\nSource counts:")
print(source_counts)

# ------------------------------------------------------------------------------
# 3. ROBUST SOURCE NORMALIZATION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. BUILD ROBUST source_is_s3 FLAG")
print("-" * 78)

cand = cand.with_columns(
    pl.col("source")
    .str.strip_chars()
    .str.to_lowercase()
    .alias("_source_norm")
)

# Anything whose normalized source clearly denotes source 3 is S3.
#
# Handles forms such as:
#   source3
#   source_3
#   s3
#   s_3
#   3
#   S3
#
# Everything else is treated as non-S3 / S2 for this 2-source training pool.

cand = cand.with_columns(
    pl.when(
        pl.col("_source_norm").str.contains(r"(source|src|s)[_\- ]*3$")
        | (pl.col("_source_norm") == "3")
    )
    .then(1)
    .otherwise(0)
    .cast(pl.Int8)
    .alias("source_is_s3")
)

source_map_check = (
    cand
    .group_by(["source", "source_is_s3"])
    .agg(pl.len().alias("rows"))
    .sort(["source", "source_is_s3"])
)

print(source_map_check)

cand = cand.drop("_source_norm")

# ------------------------------------------------------------------------------
# 4. ADD MISSING BLOCKER FLAGS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. NORMALIZE BLOCKER COLUMNS")
print("-" * 78)

for col in [
    "block_exact",
    "block_first_last",
    "block_address_exact",
    "block_address_numeric",
]:
    if col not in cand.columns:
        print(f"Adding missing {col}=0")
        cand = cand.with_columns(
            pl.lit(0, dtype=pl.Int8).alias(col)
        )
    else:
        cand = cand.with_columns(
            pl.col(col).fill_null(0).cast(pl.Int8).alias(col)
        )

# ------------------------------------------------------------------------------
# 5. DEDUP CANDIDATE PAIRS
# ------------------------------------------------------------------------------

before = cand.height

cand = (
    cand
    .unique(
        subset=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        keep="first",
        maintain_order=False,
    )
)

after = cand.height

print(f"Before dedup: {before:,}")
print(f"After dedup : {after:,}")
print(f"Removed     : {before - after:,}")

# Surviving historical artifact is expected to be unique.
# Do not hard-fail if it isn't — continue.
if before != after:
    print("⚠️ Duplicate pair keys existed; first row retained.")

# ------------------------------------------------------------------------------
# 6. LOAD OFFICIAL GROUND TRUTH
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. LOAD OFFICIAL GROUND TRUTH")
print("-" * 78)

gt_raw = pl.read_csv(
    GT_PATH,
    separator="\t",
    has_header=True,
    infer_schema_length=10000,
)

print(f"GT rows: {gt_raw.height:,}")
print(f"GT columns: {gt_raw.columns}")

assert "source1_entity_id" in gt_raw.columns
assert "matched_entity_ids" in gt_raw.columns

# Explicitly align GT key naming with candidate artifact.
gt = (
    gt_raw
    .select([
        pl.col("source1_entity_id")
        .cast(pl.Utf8)
        .alias("s1_entity_id"),

        pl.col("matched_entity_ids")
        .cast(pl.Utf8)
        .alias("matched_entity_ids"),
    ])
)

# ------------------------------------------------------------------------------
# 7. EXPAND GT TO PAIR EDGES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. EXPAND GT EDGES")
print("-" * 78)

gt_edges = (
    gt
    .with_columns(
        pl.col("matched_entity_ids")
        .str.split(",")
        .alias("candidate_entity_id")
    )
    .explode("candidate_entity_id")
    .with_columns(
        pl.col("candidate_entity_id")
        .str.strip_chars()
        .cast(pl.Utf8)
    )
    .filter(
        pl.col("candidate_entity_id").is_not_null()
        & (pl.col("candidate_entity_id") != "")
    )
    .select([
        "s1_entity_id",
        "candidate_entity_id",
    ])
    .unique(
        subset=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        maintain_order=False,
    )
    .with_columns(
        pl.lit(1, dtype=pl.Int8).alias("is_positive")
    )
)

print(f"Expanded GT edges: {gt_edges.height:,}")

assert gt_edges.height == 7_638_365, (
    f"GT edge count mismatch: {gt_edges.height:,}"
)

print("✅ Official 7,638,365-edge GT recovered.")

# ------------------------------------------------------------------------------
# 8. LABEL CANDIDATE POOL
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. LABEL CANDIDATE POOL")
print("-" * 78)

labeled = (
    cand
    .join(
        gt_edges,
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="left",
    )
    .with_columns(
        pl.col("is_positive")
        .fill_null(0)
        .cast(pl.Int8)
    )
)

candidate_positive_count = (
    labeled
    .select(pl.col("is_positive").sum())
    .item()
)

candidate_negative_count = (
    labeled.height - candidate_positive_count
)

candidate_recall_ceiling = (
    candidate_positive_count / gt_edges.height
)

print(f"Candidate rows       : {labeled.height:,}")
print(f"Positive candidates  : {candidate_positive_count:,}")
print(f"Negative candidates  : {candidate_negative_count:,}")
print(f"Candidate recall cap : {candidate_recall_ceiling:.6%}")

# ------------------------------------------------------------------------------
# 9. CANDIDATE COUNT FEATURES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("8. COMPUTE CANDIDATE COUNT FEATURES")
print("-" * 78)

source_counts = (
    labeled
    .group_by([
        "s1_entity_id",
        "source",
    ])
    .agg(
        pl.len().alias("candidate_count_source")
    )
)

total_counts = (
    labeled
    .group_by("s1_entity_id")
    .agg(
        pl.len().alias("candidate_count_total")
    )
)

labeled = (
    labeled
    .join(
        source_counts,
        on=[
            "s1_entity_id",
            "source",
        ],
        how="left",
    )
    .join(
        total_counts,
        on="s1_entity_id",
        how="left",
    )
)

# ------------------------------------------------------------------------------
# 10. DETERMINISTIC NEGATIVE SAMPLING
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("9. BUILD 3,429,214-ROW TRAINING PAIR SAMPLE")
print("-" * 78)

assert candidate_positive_count <= TARGET_ROWS, (
    f"Candidate positives ({candidate_positive_count:,}) exceed "
    f"target ({TARGET_ROWS:,})"
)

negative_needed = TARGET_ROWS - candidate_positive_count

print(f"Target rows   : {TARGET_ROWS:,}")
print(f"Positives kept: {candidate_positive_count:,}")
print(f"Negatives req : {negative_needed:,}")

# Split positives / negatives.
positive_pairs = (
    labeled
    .filter(pl.col("is_positive") == 1)
)

negative_pairs = (
    labeled
    .filter(pl.col("is_positive") == 0)
)

assert negative_needed <= negative_pairs.height

# Stable hash independent of dataframe row order.
negative_pairs = (
    negative_pairs
    .with_columns(
        pl.concat_str(
            [
                pl.lit(str(SEED)),
                pl.col("source"),
                pl.col("s1_entity_id"),
                pl.col("candidate_entity_id"),
            ],
            separator="|",
        )
        .hash(seed=SEED)
        .alias("_sample_hash")
    )
    .sort("_sample_hash")
    .head(negative_needed)
    .drop("_sample_hash")
)

sampled = pl.concat(
    [
        positive_pairs,
        negative_pairs,
    ],
    how="vertical_relaxed",
)

print(f"Sample rows before final sort: {sampled.height:,}")

assert sampled.height == TARGET_ROWS

# ------------------------------------------------------------------------------
# 11. FINAL FEATURE-ENGINEERING INPUT COLUMNS
# ------------------------------------------------------------------------------

FINAL_COLUMNS = [
    "s1_entity_id",
    "candidate_entity_id",
    "source",
    "source_is_s3",
    "block_exact",
    "block_first_last",
    "block_address_exact",
    "block_address_numeric",
    "candidate_count_source",
    "candidate_count_total",
    "is_positive",
]

for col in FINAL_COLUMNS:
    assert col in sampled.columns, f"Missing final column: {col}"

sampled = sampled.select(FINAL_COLUMNS)

# Stable ordering.
sampled = sampled.sort(
    [
        "s1_entity_id",
        "source",
        "candidate_entity_id",
    ]
)

# ------------------------------------------------------------------------------
# 12. FINAL SANITY CHECKS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("10. FINAL SANITY CHECKS")
print("-" * 78)

assert sampled.height == TARGET_ROWS

dup_pairs = (
    sampled
    .group_by([
        "s1_entity_id",
        "candidate_entity_id",
    ])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print(f"Duplicate pairs: {dup_pairs:,}")

assert dup_pairs == 0

sample_positive_count = (
    sampled
    .select(pl.col("is_positive").sum())
    .item()
)

print(f"Sample positives: {sample_positive_count:,}")
print(f"Sample negatives: {sampled.height - sample_positive_count:,}")

# Every retained positive must actually be in GT.
bad_positive_count = (
    sampled
    .filter(pl.col("is_positive") == 1)
    .join(
        gt_edges.select([
            "s1_entity_id",
            "candidate_entity_id",
        ]),
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="anti",
    )
    .height
)

print(f"Bad positive labels: {bad_positive_count:,}")

assert bad_positive_count == 0

# ------------------------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("11. SAVE PAIR SAMPLE")
print("-" * 78)

if BASELINE_PAIR_PATH.exists():
    BASELINE_PAIR_PATH.unlink()

sampled.write_parquet(
    BASELINE_PAIR_PATH,
    compression="zstd",
    statistics=True,
)

print(f"Saved: {BASELINE_PAIR_PATH}")

# ------------------------------------------------------------------------------
# 14. RELOAD CHECK
# ------------------------------------------------------------------------------

check = pl.read_parquet(BASELINE_PAIR_PATH)

print(f"Reloaded rows: {check.height:,}")

assert check.height == TARGET_ROWS
assert check.columns == FINAL_COLUMNS

# ------------------------------------------------------------------------------
# 15. CREATE DOWNSTREAM GLOBALS
# ------------------------------------------------------------------------------

# These are deliberately exported so the next feature-engineering cell can
# consume this bridge without requiring the old Cell-58 Python state.

BASELINE_ROOT = Path(
    "/kaggle/working/AMLC2026/submission_01_baseline"
)
BASELINE_ROOT.mkdir(parents=True, exist_ok=True)

positive_edges = gt_edges.select([
    "s1_entity_id",
    "candidate_entity_id",
])

# Original validation convention:
# hash(S1 ID, seed=2026) % 10 == 0
#
# We construct this from the S1 universe rather than storing 200k+ Python
# objects unnecessarily.
all_s1 = (
    gt
    .select("s1_entity_id")
    .unique()
)

val_s1_df = (
    all_s1
    .with_columns(
        pl.col("s1_entity_id")
        .hash(seed=SEED)
        .mod(10)
        .alias("_fold")
    )
    .filter(pl.col("_fold") == 0)
    .select("s1_entity_id")
)

# A Python set is convenient for downstream membership checks.
val_s1 = set(
    val_s1_df
    .get_column("s1_entity_id")
    .to_list()
)

print(f"\nValidation S1 count: {len(val_s1):,}")

# ------------------------------------------------------------------------------
# 16. SAVE METADATA
# ------------------------------------------------------------------------------

metadata = {
    "cell": "59",
    "mode": "self-contained-pair-sample-bridge",
    "seed": SEED,

    "candidate_path": str(CANDIDATE_PATH),
    "gt_path": str(GT_PATH),
    "output_path": str(BASELINE_PAIR_PATH),

    "candidate_rows": int(cand.height),
    "gt_edges": int(gt_edges.height),

    "candidate_positive_count": int(candidate_positive_count),
    "candidate_negative_count": int(candidate_negative_count),

    "candidate_recall_ceiling": float(candidate_recall_ceiling),

    "target_sample_rows": int(TARGET_ROWS),
    "sample_positive_count": int(sample_positive_count),
    "sample_negative_count": int(
        TARGET_ROWS - sample_positive_count
    ),

    "validation_s1_count": int(len(val_s1)),

    "source_values": [
        str(x)
        for x in source_values.get_column("source").to_list()
    ],

    "note": (
        "Original Cell-58 candidate artifact preserved its own source "
        "encoding. No strict source2/source3 assertion is used."
    ),
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------------------------
# 17. FINAL
# ------------------------------------------------------------------------------

elapsed = (time.time() - T0) / 60

print("\n" + "=" * 78)
print("✅ CELL 59 BRIDGE COMPLETE")
print("=" * 78)

print(f"""
PAIR SAMPLE:
  {BASELINE_PAIR_PATH}

ROWS:
  {check.height:,}

POSITIVES:
  {sample_positive_count:,}

NEGATIVES:
  {check.height - sample_positive_count:,}

GT EDGES:
  {gt_edges.height:,}

CANDIDATE RECALL CEILING:
  {candidate_recall_ceiling:.6%}

SOURCE VALUES:
  {[str(x) for x in source_values.get_column("source").to_list()]}

VALIDATION S1:
  {len(val_s1):,}

RUNTIME:
  {elapsed:.2f} min

✅ No source-value assertion.
✅ Official GT joined.
✅ Exact 3,429,214 rows produced.
✅ Pair uniqueness verified.
✅ Positive-label audit passed.
✅ BASELINE_PAIR_PATH exported.
✅ positive_edges exported.
✅ val_s1 exported.
""")

AMLC 2026 — CELL 59 — SELF-CONTAINED PAIR-SAMPLE BRIDGE

Candidate : /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/state/eval_candidates_tier12_source_capped.parquet
GT        : /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_ground_truth.tsv
Output    : /kaggle/working/AMLC2026/pair_sample_cell58/baseline_candidate_pairs.parquet

------------------------------------------------------------------------------
1. LOAD SURVIVING CELL-58 CANDIDATE POOL
------------------------------------------------------------------------------
Rows: 9,494,493
Columns: ['s1_entity_id', 'candidate_entity_id', 'source', 'block_exact', 'block_first_last']

------------------------------------------------------------------------------
2. ACTUAL SOURCE VALUES
------------------------------------------------------------------------------
shape: (2, 1)
┌────────┐
│ source │
│ --

In [6]:
# ==============================================================================
# AMLC 2026 — CHECKPOINT AFTER CELL 59 BRIDGE
# Exact resumable state before CELL 61
# ==============================================================================

from pathlib import Path
import json
import hashlib
import shutil
import subprocess
import sys
import time

import polars as pl


print("=" * 78)
print("AMLC 2026 — CHECKPOINT AFTER CELL 59 BRIDGE")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# SOURCE STATE
# ------------------------------------------------------------------------------

MASTER_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL"
)

PAIR_SAMPLE_ROOT = Path(
    "/kaggle/working/AMLC2026/pair_sample_cell58"
)

BASELINE_PAIR_PATH = (
    PAIR_SAMPLE_ROOT / "baseline_candidate_pairs.parquet"
)

GT_PATH = (
    MASTER_ROOT
    / "dataset"
    / "train"
    / "train_ground_truth.tsv"
)

# ------------------------------------------------------------------------------
# CHECKPOINT LOCATION
# ------------------------------------------------------------------------------

CHECKPOINT_ROOT = Path(
    "/kaggle/working/AMLC2026/checkpoint_after_cell59_BRIDGE_20260925"
)

if CHECKPOINT_ROOT.exists():
    shutil.rmtree(CHECKPOINT_ROOT)

CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

STATE_ROOT = CHECKPOINT_ROOT / "state"
STATE_ROOT.mkdir(parents=True, exist_ok=True)

# Portable zip goes beside checkpoint.
CHECKPOINT_ZIP = Path(
    "/kaggle/working/AMLC2026/AMLC2026_AFTER_CELL59_BRIDGE_20260925.zip"
)

if CHECKPOINT_ZIP.exists():
    CHECKPOINT_ZIP.unlink()

print(f"\nCheckpoint root:\n{CHECKPOINT_ROOT}")

# ------------------------------------------------------------------------------
# 1. REQUIRED LIVE ARTIFACT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. VERIFY CURRENT CELL-59 STATE")
print("-" * 78)

assert BASELINE_PAIR_PATH.exists(), (
    f"Missing pair sample:\n{BASELINE_PAIR_PATH}"
)

pair_check = pl.read_parquet(BASELINE_PAIR_PATH)

print(f"Pair rows   : {pair_check.height:,}")
print(f"Pair columns: {pair_check.columns}")

assert pair_check.height == 3_429_214

required_pair_columns = [
    "s1_entity_id",
    "candidate_entity_id",
    "source",
    "source_is_s3",
    "block_exact",
    "block_first_last",
    "block_address_exact",
    "block_address_numeric",
    "candidate_count_source",
    "candidate_count_total",
    "is_positive",
]

assert pair_check.columns == required_pair_columns

# ------------------------------------------------------------------------------
# 2. RECOVER / VERIFY POSITIVE EDGES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. BUILD CHECKPOINT POSITIVE-EDGE TABLE")
print("-" * 78)

if "positive_edges" in globals():
    positive_edges_ckpt = positive_edges.clone()
else:
    positive_edges_ckpt = (
        pair_check
        .filter(pl.col("is_positive") == 1)
        .select([
            "s1_entity_id",
            "candidate_entity_id",
        ])
    )

print(f"Positive edges available in checkpoint: {positive_edges_ckpt.height:,}")

# IMPORTANT:
# The official full GT has 7,638,365 edges. The pair sample contains only
# candidate positives, so we preserve BOTH:
#   - full official GT edge table
#   - Cell-59 sampled positive edge table

GT_EDGES_PATH = STATE_ROOT / "gt_edges_full.parquet"

print("Re-expanding official GT for durable checkpoint...")

gt_raw = pl.read_csv(
    GT_PATH,
    separator="\t",
    has_header=True,
    infer_schema_length=10000,
)

gt_edges_full = (
    gt_raw
    .select([
        pl.col("source1_entity_id")
        .cast(pl.Utf8)
        .alias("s1_entity_id"),

        pl.col("matched_entity_ids")
        .cast(pl.Utf8)
        .alias("matched_entity_ids"),
    ])
    .with_columns(
        pl.col("matched_entity_ids")
        .str.split(",")
        .alias("candidate_entity_id")
    )
    .explode("candidate_entity_id")
    .with_columns(
        pl.col("candidate_entity_id")
        .str.strip_chars()
        .cast(pl.Utf8)
    )
    .filter(
        pl.col("candidate_entity_id").is_not_null()
        & (pl.col("candidate_entity_id") != "")
    )
    .select([
        "s1_entity_id",
        "candidate_entity_id",
    ])
    .unique(
        subset=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        maintain_order=False,
    )
)

print(f"Full GT edges: {gt_edges_full.height:,}")

assert gt_edges_full.height == 7_638_365

gt_edges_full.write_parquet(
    GT_EDGES_PATH,
    compression="zstd",
    statistics=True,
)

# Sample-positive table.
SAMPLE_POSITIVE_PATH = STATE_ROOT / "sample_positive_edges.parquet"

positive_edges_ckpt.write_parquet(
    SAMPLE_POSITIVE_PATH,
    compression="zstd",
    statistics=True,
)

# ------------------------------------------------------------------------------
# 3. COPY THE EXACT PAIR SAMPLE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. COPY EXACT 3,429,214-ROW PAIR SAMPLE")
print("-" * 78)

PAIR_SAMPLE_CKPT = (
    STATE_ROOT / "baseline_candidate_pairs.parquet"
)

shutil.copy2(
    BASELINE_PAIR_PATH,
    PAIR_SAMPLE_CKPT,
)

print(f"Saved:\n{PAIR_SAMPLE_CKPT}")

# ------------------------------------------------------------------------------
# 4. SAVE VALIDATION S1
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. SAVE VALIDATION S1")
print("-" * 78)

VAL_S1_PATH = STATE_ROOT / "val_s1.parquet"

if "val_s1" in globals():

    val_s1_list = sorted(
        str(x)
        for x in val_s1
    )

    val_s1_df = pl.DataFrame({
        "s1_entity_id": val_s1_list
    })

else:

    # Reconstruct from official GT S1 IDs using the same hash convention.
    all_s1 = (
        gt_edges_full
        .select("s1_entity_id")
        .unique()
    )

    val_s1_df = (
        all_s1
        .with_columns(
            pl.col("s1_entity_id")
            .hash(seed=2026)
            .mod(10)
            .alias("_fold")
        )
        .filter(pl.col("_fold") == 0)
        .select("s1_entity_id")
    )

val_s1_df = val_s1_df.unique().sort("s1_entity_id")

print(f"Validation S1 rows: {val_s1_df.height:,}")

val_s1_df.write_parquet(
    VAL_S1_PATH,
    compression="zstd",
)

# ------------------------------------------------------------------------------
# 5. SAVE EXACT STATE METADATA
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. SAVE STATE METADATA")
print("-" * 78)

candidate_recall_ceiling = (
    pair_check
    .filter(pl.col("is_positive") == 1)
    .height
    / 7_638_365.0
)

sample_positive_count = (
    pair_check
    .select(pl.col("is_positive").sum())
    .item()
)

sample_negative_count = (
    pair_check.height - sample_positive_count
)

source_values = sorted(
    str(x)
    for x in
    pair_check
    .select("source")
    .unique()
    .get_column("source")
    .to_list()
)

metadata = {
    "checkpoint_name":
        "AMLC2026_AFTER_CELL59_BRIDGE_20260925",

    "created_at":
        time.strftime("%Y-%m-%d %H:%M:%S"),

    "seed":
        2026,

    "target_pair_rows":
        3_429_214,

    "pair_rows":
        int(pair_check.height),

    "pair_positive_rows":
        int(sample_positive_count),

    "pair_negative_rows":
        int(sample_negative_count),

    "full_gt_edges":
        int(gt_edges_full.height),

    "sample_candidate_recall_ceiling":
        float(candidate_recall_ceiling),

    "validation_s1_rows":
        int(val_s1_df.height),

    "source_values":
        source_values,

    "master_root":
        str(MASTER_ROOT),

    "candidate_source_artifact":
        str(
            MASTER_ROOT
            / "checkpoint_after_cell58_FINAL_20260925"
            / "state"
            / "eval_candidates_tier12_source_capped.parquet"
        ),

    "ground_truth":
        str(GT_PATH),

    "pair_sample":
        "state/baseline_candidate_pairs.parquet",

    "full_gt_edges_file":
        "state/gt_edges_full.parquet",

    "sample_positive_edges_file":
        "state/sample_positive_edges.parquet",

    "validation_s1_file":
        "state/val_s1.parquet",

    "next_cell":
        "CELL 61",
}

METADATA_PATH = CHECKPOINT_ROOT / "CHECKPOINT_METADATA.json"

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

# ------------------------------------------------------------------------------
# 6. ENVIRONMENT MANIFEST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. SAVE ENVIRONMENT MANIFEST")
print("-" * 78)

ENV_PATH = CHECKPOINT_ROOT / "environment.txt"

try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        timeout=120,
    )

    ENV_PATH.write_text(result.stdout)

    print(f"Saved: {ENV_PATH}")

except Exception as e:
    ENV_PATH.write_text(
        f"pip freeze failed: {repr(e)}\n"
    )
    print(f"⚠️ Could not capture pip freeze: {e}")

# ------------------------------------------------------------------------------
# 7. SHA256 MANIFEST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. CREATE SHA256 MANIFEST")
print("-" * 78)

def sha256_file(path: Path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


manifest = {}

for path in CHECKPOINT_ROOT.rglob("*"):
    if path.is_file():
        rel = str(path.relative_to(CHECKPOINT_ROOT))
        manifest[rel] = {
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }

MANIFEST_PATH = CHECKPOINT_ROOT / "SHA256_MANIFEST.json"

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2, sort_keys=True)

print(f"Manifest entries: {len(manifest)}")
print(f"Saved: {MANIFEST_PATH}")

# ------------------------------------------------------------------------------
# 8. PORTABLE ZIP
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("8. BUILD PORTABLE ZIP")
print("-" * 78)

archive_base = CHECKPOINT_ZIP.with_suffix("")

shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=CHECKPOINT_ROOT,
)

assert CHECKPOINT_ZIP.exists()

zip_sha256 = sha256_file(CHECKPOINT_ZIP)

ZIP_SHA_PATH = (
    CHECKPOINT_ROOT / "CHECKPOINT_ZIP_SHA256.txt"
)

ZIP_SHA_PATH.write_text(
    zip_sha256 + "\n"
)

# ------------------------------------------------------------------------------
# 9. HARD RELOAD VERIFICATION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("9. HARD RELOAD VERIFICATION")
print("-" * 78)

pair_reload = pl.read_parquet(
    PAIR_SAMPLE_CKPT
)

gt_reload = pl.read_parquet(
    GT_EDGES_PATH
)

val_reload = pl.read_parquet(
    VAL_S1_PATH
)

assert pair_reload.height == 3_429_214
assert gt_reload.height == 7_638_365
assert val_reload.height == val_s1_df.height

print(f"Pair sample reload : {pair_reload.height:,}")
print(f"GT reload          : {gt_reload.height:,}")
print(f"Val S1 reload      : {val_reload.height:,}")

# ------------------------------------------------------------------------------
# 10. FINAL
# ------------------------------------------------------------------------------

elapsed = (time.time() - T0) / 60

print("\n" + "=" * 78)
print("✅ CHECKPOINT AFTER CELL 59 CREATED")
print("=" * 78)

print(f"""
CHECKPOINT:
  {CHECKPOINT_ROOT}

PORTABLE ZIP:
  {CHECKPOINT_ZIP}

ZIP SHA256:
  {zip_sha256}

EXACT PAIR SAMPLE:
  {PAIR_SAMPLE_CKPT}
  rows = {pair_reload.height:,}

FULL GT:
  {GT_EDGES_PATH}
  rows = {gt_reload.height:,}

VALIDATION S1:
  {VAL_S1_PATH}
  rows = {val_reload.height:,}

METADATA:
  {METADATA_PATH}

✅ Pair sample survives reload.
✅ Full 7,638,365 GT edges survive reload.
✅ Validation S1 survives reload.
✅ SHA256 manifest created.
✅ Portable ZIP created.

NEXT:
  Resume from this checkpoint, then run CELL 61.
  
Runtime: {elapsed:.2f} minutes
""")

AMLC 2026 — CHECKPOINT AFTER CELL 59 BRIDGE

Checkpoint root:
/kaggle/working/AMLC2026/checkpoint_after_cell59_BRIDGE_20260925

------------------------------------------------------------------------------
1. VERIFY CURRENT CELL-59 STATE
------------------------------------------------------------------------------
Pair rows   : 3,429,214
Pair columns: ['s1_entity_id', 'candidate_entity_id', 'source', 'source_is_s3', 'block_exact', 'block_first_last', 'block_address_exact', 'block_address_numeric', 'candidate_count_source', 'candidate_count_total', 'is_positive']

------------------------------------------------------------------------------
2. BUILD CHECKPOINT POSITIVE-EDGE TABLE
------------------------------------------------------------------------------
Positive edges available in checkpoint: 7,638,365
Re-expanding official GT for durable checkpoint...
Full GT edges: 7,638,365

------------------------------------------------------------------------------
3. COPY EXACT 3,429,214-

In [7]:
# ==============================================================================
# CELL 61 — AMLC 2026 FINAL FEATURE LAKE BOOTSTRAP
# ==============================================================================
# Purpose:
#   1. Freeze paths/versioning
#   2. Verify Kaggle GPU
#   3. Create permanent feature-lake directories
#   4. Record the exact environment
# ==============================================================================

from pathlib import Path
import os
import json
import hashlib
import platform
import subprocess
import time

import numpy as np
import pandas as pd
import polars as pl
import torch

# ------------------------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------------------------

MASTER_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/"
    "amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL"
)

TRAIN_ROOT = MASTER_ROOT / "dataset" / "train"
TEST_ROOT  = MASTER_ROOT / "dataset" / "test"

WORK_ROOT = Path("/kaggle/working/AMLC2026")

FEATURE_ROOT = WORK_ROOT / "FINAL_FEATURE_LAKE_V1"

RECORD_ROOT      = FEATURE_ROOT / "records"
CANDIDATE_ROOT   = FEATURE_ROOT / "candidates"
FEATURE_TABLE_ROOT = FEATURE_ROOT / "features"
EMBED_ROOT       = FEATURE_ROOT / "embeddings"
META_ROOT        = FEATURE_ROOT / "metadata"
LOG_ROOT         = FEATURE_ROOT / "logs"
SCHEMA_ROOT      = FEATURE_ROOT / "schema"

for p in [
    FEATURE_ROOT,
    RECORD_ROOT,
    CANDIDATE_ROOT,
    FEATURE_TABLE_ROOT,
    EMBED_ROOT,
    META_ROOT,
    LOG_ROOT,
    SCHEMA_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

for source in ["s1", "s2", "s3"]:
    (RECORD_ROOT / "train" / source).mkdir(parents=True, exist_ok=True)
    (RECORD_ROOT / "test" / source).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. GPU CHECK
# ------------------------------------------------------------------------------

print("=" * 78)
print("AMLC 2026 — FINAL FEATURE LAKE V1")
print("=" * 78)

print("MASTER_ROOT:", MASTER_ROOT)
print("TRAIN_ROOT :", TRAIN_ROOT)
print("TEST_ROOT  :", TEST_ROOT)
print("FEATURE_ROOT:", FEATURE_ROOT)

assert MASTER_ROOT.exists(), f"Missing MASTER_ROOT: {MASTER_ROOT}"
assert TRAIN_ROOT.exists(), f"Missing TRAIN_ROOT: {TRAIN_ROOT}"
assert TEST_ROOT.exists(), f"Missing TEST_ROOT: {TEST_ROOT}"

print("\n" + "=" * 78)
print("GPU")
print("=" * 78)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is NOT available. Stop here. We explicitly want the T4 used "
        "for semantic embedding generation."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)

print("GPU:", gpu_name)
print("VRAM GB:", round(gpu_props.total_memory / 1024**3, 2))
print("CUDA runtime:", torch.version.cuda)

# Small actual GPU computation.
x = torch.randn((4096, 4096), device="cuda", dtype=torch.float16)
y = x @ x.T
torch.cuda.synchronize()

print("GPU matrix smoke:", y.shape, y.dtype)
del x, y
torch.cuda.empty_cache()

# ------------------------------------------------------------------------------
# 3. CPU / RAM
# ------------------------------------------------------------------------------

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1024**3
    print("System RAM GB:", round(ram_gb, 2))
except Exception:
    print("psutil unavailable")

# ------------------------------------------------------------------------------
# 4. VERSION MANIFEST
# ------------------------------------------------------------------------------

manifest = {
    "feature_lake_version": "V1",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "master_root": str(MASTER_ROOT),
    "train_root": str(TRAIN_ROOT),
    "test_root": str(TEST_ROOT),
    "gpu": gpu_name,
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "python_version": platform.python_version(),
    "feature_plan": "415-column canonical schema",
    "semantic_model": "intfloat/multilingual-e5-small",
    "semantic_dimension": 384,
    "semantic_dtype": "float16",
}

with open(SCHEMA_ROOT / "feature_lake_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\n✅ CELL 61 PASSED")
print("Feature lake:", FEATURE_ROOT)

AMLC 2026 — FINAL FEATURE LAKE V1
MASTER_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL
TRAIN_ROOT : /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train
TEST_ROOT  : /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/test
FEATURE_ROOT: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

GPU
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM GB: 14.56
CUDA runtime: 12.8
GPU matrix smoke: torch.Size([4096, 4096]) torch.float16
System RAM GB: 31.35

✅ CELL 61 PASSED
Feature lake: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1


In [8]:
pip install anyascii

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 11.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [9]:
# ==============================================================================
# CELL 62 — CANONICAL RECORD TABLE BUILDER
# ==============================================================================
# Creates:
#
#   records/train/s1/records.parquet
#   records/train/s2/records.parquet
#   records/train/s3/records.parquet
#   records/test/s1/records.parquet
#   records/test/s2/records.parquet
#   records/test/s3/records.parquet
#
# These are the canonical inputs for the SAME feature factory used by
# training and inference.
# ==============================================================================


import re
import unicodedata
import anyascii
import polars as pl
from pathlib import Path

# ------------------------------------------------------------------------------
# NORMALIZATION CONSTANTS
# ------------------------------------------------------------------------------

LEGAL_SUFFIX_RE = re.compile(
    r"""
    (?:
        \bprivate\s+limited\b |
        \bprivate\b |
        \blimited\b |
        \bltd\b |
        \bllp\b |
        \bllc\b |
        \bincorporated\b |
        \binc\b |
        \bcorporation\b |
        \bcorp\b |
        \bcompany\b |
        \bco\b |
        \bplc\b |
        \bgmbh\b |
        \bsarl\b |
        \bbv\b |
        \bag\b |
        \bspa\b
    )
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

# ------------------------------------------------------------------------------
# PYTHON NORMALIZATION
# Used only on unique strings or low-volume transformations.
# Main canonical normalization below is vectorized in Polars.
# ------------------------------------------------------------------------------

def canonical_text_py(x):
    if x is None:
        return ""

    x = str(x)
    x = unicodedata.normalize("NFKC", x)
    x = x.casefold()
    x = x.replace("&", " and ")

    # Keep Unicode letters/digits; normalize punctuation to spaces.
    chars = []
    for ch in x:
        cat = unicodedata.category(ch)
        if ch.isalnum():
            chars.append(ch)
        else:
            chars.append(" ")

    x = "".join(chars)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def suffix_removed_py(x):
    x = canonical_text_py(x)
    if not x:
        return ""

    x = LEGAL_SUFFIX_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def ascii_text_py(x):
    if not x:
        return ""
    return anyascii.anyascii(x).casefold().strip()


def translit_text_py(x):
    # anyascii is our deterministic auxiliary transliteration.
    return ascii_text_py(x)


def token_list_py(x):
    if not x:
        return []
    return x.split()


def digit_signature_py(x):
    if not x:
        return ""

    vals = re.findall(r"\d+[A-Za-z]*", x)
    return " ".join(vals)


def alpha_signature_py(x):
    if not x:
        return ""

    vals = re.findall(r"[A-Za-z\u00C0-\uFFFF]+", x)
    return " ".join(vals)


# ------------------------------------------------------------------------------
# POLARS FEATURE CONSTRUCTION
# ------------------------------------------------------------------------------

def build_record_table(path: Path, source: str, split: str):
    print("\n" + "=" * 78)
    print(f"{split.upper()} / {source.upper()}")
    print("=" * 78)
    print("Reading:", path)

    df = pl.read_csv(
        path,
        separator="\t",
        infer_schema_length=10000,
        ignore_errors=False,
        null_values=["", "NULL", "null", "None"],
    )

    # Normalize possible accidental whitespace in column names.
    df = df.rename({c: c.strip() for c in df.columns})

    required = {
        "entity_id",
        "business_name",
        "business_address",
        "country",
    }

    missing = required - set(df.columns)
    if missing:
        raise RuntimeError(
            f"{path} missing required columns: {sorted(missing)}\n"
            f"Columns found: {df.columns}"
        )

    df = df.select(
        [
            pl.col("entity_id").cast(pl.Utf8),
            pl.col("business_name").cast(pl.Utf8).fill_null(""),
            pl.col("business_address").cast(pl.Utf8).fill_null(""),
            pl.col("country").cast(pl.Utf8).fill_null(""),
        ]
    )

    # --------------------------------------------------------------------------
    # Canonical normalized fields
    # --------------------------------------------------------------------------

    df = df.with_columns(
        [
            pl.col("business_name")
              .map_elements(canonical_text_py, return_dtype=pl.Utf8)
              .alias("name_norm"),

            pl.col("business_address")
              .map_elements(canonical_text_py, return_dtype=pl.Utf8)
              .alias("address_norm"),

            pl.col("business_name")
              .map_elements(suffix_removed_py, return_dtype=pl.Utf8)
              .alias("name_suffix_removed"),

            pl.col("business_address")
              .map_elements(ascii_text_py, return_dtype=pl.Utf8)
              .alias("address_ascii"),
        ]
    )

    df = df.with_columns(
        [
            pl.col("name_norm")
              .map_elements(ascii_text_py, return_dtype=pl.Utf8)
              .alias("name_ascii"),

            pl.col("name_norm")
              .map_elements(translit_text_py, return_dtype=pl.Utf8)
              .alias("name_translit"),

            pl.col("address_norm")
              .map_elements(translit_text_py, return_dtype=pl.Utf8)
              .alias("address_translit"),

            pl.col("name_norm")
              .map_elements(digit_signature_py, return_dtype=pl.Utf8)
              .alias("name_digit_signature"),

            pl.col("address_norm")
              .map_elements(digit_signature_py, return_dtype=pl.Utf8)
              .alias("address_digit_signature"),

            pl.col("name_norm")
              .map_elements(alpha_signature_py, return_dtype=pl.Utf8)
              .alias("name_alpha_signature"),

            pl.col("address_norm")
              .map_elements(alpha_signature_py, return_dtype=pl.Utf8)
              .alias("address_alpha_signature"),
        ]
    )

    # --------------------------------------------------------------------------
    # Cheap record-level structural features
    # --------------------------------------------------------------------------

    df = df.with_columns(
        [
            pl.col("name_norm").str.len_chars().cast(pl.Int32).alias("name_len"),
            pl.col("address_norm").str.len_chars().cast(pl.Int32).alias("address_len"),

            pl.col("name_norm")
              .str.count_matches(r"\S+")
              .cast(pl.Int16)
              .alias("name_token_count"),

            pl.col("address_norm")
              .str.count_matches(r"\S+")
              .cast(pl.Int16)
              .alias("address_token_count"),

            pl.col("name_norm")
              .str.count_matches(r"\d")
              .cast(pl.Int16)
              .alias("name_digit_count"),

            pl.col("address_norm")
              .str.count_matches(r"\d")
              .cast(pl.Int16)
              .alias("address_digit_count"),

            pl.col("name_norm")
              .str.count_matches(r"[^\x00-\x7F]")
              .cast(pl.Int16)
              .alias("name_nonascii_count"),

            pl.col("address_norm")
              .str.count_matches(r"[^\x00-\x7F]")
              .cast(pl.Int16)
              .alias("address_nonascii_count"),
        ]
    )

    # --------------------------------------------------------------------------
    # Token / first-last representations
    # --------------------------------------------------------------------------

    df = df.with_columns(
        [
            pl.col("name_norm")
              .str.split(" ")
              .list.first()
              .fill_null("")
              .alias("name_first_token"),

            pl.col("name_norm")
              .str.split(" ")
              .list.last()
              .fill_null("")
              .alias("name_last_token"),
        ]
    )

    df = df.with_columns(
        [
            (
                pl.col("name_first_token") + pl.lit(" ") +
                pl.col("name_last_token")
            ).str.strip_chars().alias("name_first_last"),

            pl.col("name_norm")
              .str.split(" ")
              .list.eval(pl.element().str.slice(0, 1))
              .list.join("")
              .alias("name_initials"),
        ]
    )

    # --------------------------------------------------------------------------
    # Full-record semantic text
    # --------------------------------------------------------------------------

    df = df.with_columns(
        (
            pl.lit("name: ") + pl.col("name_norm") +
            pl.lit(" address: ") + pl.col("address_norm") +
            pl.lit(" country: ") + pl.col("country")
        ).alias("full_record_text")
    )

    # --------------------------------------------------------------------------
    # Source metadata
    # --------------------------------------------------------------------------

    df = df.with_columns(
        [
            pl.lit(source).alias("source"),
            pl.lit(split).alias("split"),

            (pl.col("business_name").str.len_chars() == 0)
                .cast(pl.UInt8)
                .alias("name_missing"),

            (pl.col("business_address").str.len_chars() == 0)
                .cast(pl.UInt8)
                .alias("address_missing"),
        ]
    )

    # --------------------------------------------------------------------------
    # Uniqueness checks
    # --------------------------------------------------------------------------

    n = df.height
    unique_ids = df.select(pl.col("entity_id").n_unique()).item()

    if n != unique_ids:
        raise RuntimeError(
            f"{split}/{source}: entity_id duplicates detected: "
            f"{n} rows vs {unique_ids} unique IDs"
        )

    out = RECORD_ROOT / split / source / "records.parquet"

    df.write_parquet(
        out,
        compression="zstd",
        compression_level=3,
        statistics=True,
    )

    print("Rows:", n)
    print("Unique IDs:", unique_ids)
    print("Saved:", out)

    return df


# ------------------------------------------------------------------------------
# BUILD ALL SIX TABLES
# ------------------------------------------------------------------------------

record_tables = {}

for split, root in [
    ("train", TRAIN_ROOT),
    ("test", TEST_ROOT),
]:
    for source in ["s1", "s2", "s3"]:

        src_file = root / (
            "train_source1.tsv" if source == "s1" and split == "train"
            else "train_source2.tsv" if source == "s2" and split == "train"
            else "train_source3.tsv" if source == "s3" and split == "train"
            else "test_source1.tsv" if source == "s1"
            else "test_source2.tsv" if source == "s2"
            else "test_source3.tsv"
        )

        record_tables[(split, source)] = build_record_table(
            src_file,
            source,
            split,
        )

print("\n" + "=" * 78)
print("✅ CELL 62 COMPLETE")
print("=" * 78)

for key, df in record_tables.items():
    print(f"{key}: {df.height:,} rows × {df.width} columns")


TRAIN / S1
Reading: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_source1.tsv
Rows: 2206821
Unique IDs: 2206821
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s1/records.parquet

TRAIN / S2
Reading: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_source2.tsv
Rows: 5034616
Unique IDs: 5034616
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s2/records.parquet

TRAIN / S3
Reading: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_source3.tsv
Rows: 5285603
Unique IDs: 5285603
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s3/records.parquet

TEST / S1
Reading: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/test/test_source1.tsv
Rows: 1732544
Unique IDs: 1732544
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/

In [10]:
# ==============================================================================
# CELL 63 — FINAL FEATURE SCHEMA REGISTRY
# ==============================================================================

FEATURE_SCHEMA = {

    # --------------------------------------------------------------------------
    # Pair metadata
    # --------------------------------------------------------------------------
    "source_pair_s1_s2": "uint8",
    "source_pair_s1_s3": "uint8",
    "candidate_source_is_s2": "uint8",
    "candidate_source_is_s3": "uint8",

    "country_equal": "uint8",
    "country_left_missing": "uint8",
    "country_right_missing": "uint8",
    "country_both_present": "uint8",
    "country_mismatch": "uint8",

    "candidate_count_source": "int32",
    "candidate_count_total": "int32",
    "candidate_count_source_log1p": "float32",

    # --------------------------------------------------------------------------
    # Name exact / transform
    # --------------------------------------------------------------------------
    "name_raw_exact": "uint8",
    "name_norm_exact": "uint8",
    "name_ascii_exact": "uint8",
    "name_translit_exact": "uint8",
    "name_suffix_removed_exact": "uint8",
    "name_token_sorted_exact": "uint8",
    "name_first_last_exact": "uint8",
    "name_initials_exact": "uint8",
    "name_acronym_exact": "uint8",
    "name_digit_signature_exact": "uint8",
    "name_alpha_signature_exact": "uint8",
    "name_transform_any_exact": "uint8",
    "name_transform_count_exact": "int16",

    # --------------------------------------------------------------------------
    # Name structure
    # --------------------------------------------------------------------------
    "name_len_left": "int32",
    "name_len_right": "int32",
    "name_len_diff": "int32",
    "name_len_abs_diff": "int32",
    "name_len_ratio": "float32",

    "name_token_count_left": "int16",
    "name_token_count_right": "int16",
    "name_token_count_diff": "int16",
    "name_token_count_ratio": "float32",

    "name_unique_token_count_left": "int16",
    "name_unique_token_count_right": "int16",
    "name_unique_token_count_diff": "int16",

    "name_digit_count_diff": "int16",
    "name_alpha_count_diff": "int16",
    "name_nonascii_count_diff": "int16",

    "name_first_token_equal": "uint8",
    "name_last_token_equal": "uint8",
    "name_first_last_equal": "uint8",
    "name_first_token_similarity": "float32",
    "name_last_token_similarity": "float32",
    "name_prefix_similarity": "float32",
    "name_suffix_similarity": "float32",

    # --------------------------------------------------------------------------
    # Name similarities
    # --------------------------------------------------------------------------
    "name_ratio": "float32",
    "name_partial_ratio": "float32",
    "name_token_sort_ratio": "float32",
    "name_token_set_ratio": "float32",
    "name_weighted_ratio": "float32",
    "name_jaro": "float32",
    "name_jaro_winkler": "float32",
    "name_levenshtein_similarity": "float32",
    "name_normalized_edit_distance": "float32",
    "name_damerau_similarity": "float32",
    "name_lcs_similarity": "float32",
    "name_indel_similarity": "float32",

    "name_best_transform_ratio": "float32",
    "name_best_transform_jaro": "float32",
    "name_transform_gain_ratio": "float32",
    "name_ascii_gain_ratio": "float32",
    "name_translit_gain_ratio": "float32",
    "name_suffix_gain_ratio": "float32",

    # --------------------------------------------------------------------------
    # Name token features
    # --------------------------------------------------------------------------
    "name_token_jaccard": "float32",
    "name_token_dice": "float32",
    "name_token_overlap_count": "int16",
    "name_token_overlap_fraction_left": "float32",
    "name_token_overlap_fraction_right": "float32",
    "name_token_containment_left": "float32",
    "name_token_containment_right": "float32",
    "name_token_containment_max": "float32",
    "name_token_containment_min": "float32",
    "name_common_unique_token_count": "int16",
    "name_token_order_similarity": "float32",
    "name_token_reverse_order_similarity": "float32",
    "name_token_sequence_similarity": "float32",
    "name_initial_similarity": "float32",
    "name_acronym_similarity": "float32",
    "name_abbreviation_compatibility": "float32",
    "name_token_length_similarity": "float32",
    "name_rare_token_count_shared": "int16",
    "name_rare_token_fraction_shared": "float32",
    "name_weighted_token_jaccard": "float32",
    "name_weighted_token_dice": "float32",

    # --------------------------------------------------------------------------
    # Address / numeric / semantic families
    # --------------------------------------------------------------------------
    # We register the remaining names from the canonical list explicitly.
}

# These names will be appended from the frozen feature specification.
# Keeping this separate makes future schema validation easy.

FEATURE_GROUPS = {
    "pair_metadata": [],
    "name_exact": [],
    "name_structure": [],
    "name_similarity": [],
    "name_tokens": [],
    "name_ngrams": [],
    "address_exact": [],
    "address_structure": [],
    "address_similarity": [],
    "address_tokens": [],
    "address_ngrams": [],
    "address_numeric": [],
    "address_components": [],
    "frequency_idf": [],
    "cross_field": [],
    "blocker": [],
    "competition": [],
    "record_quality": [],
    "source_specific": [],
    "graph": [],
    "conflict": [],
    "semantic": [],
    "reranker": [],
    "fellegi_sunter": [],
    "aggregates": [],
}


# For now, use the registry as a validation contract.
schema_payload = {
    "version": "FINAL_FEATURE_SCHEMA_V1",
    "feature_count_registered": len(FEATURE_SCHEMA),
    "features": FEATURE_SCHEMA,
    "groups": FEATURE_GROUPS,
}

schema_path = SCHEMA_ROOT / "feature_schema_v1.json"

with open(schema_path, "w") as f:
    json.dump(schema_payload, f, indent=2)

print("=" * 78)
print("FEATURE SCHEMA V1")
print("=" * 78)
print("Registered columns:", len(FEATURE_SCHEMA))
print("Schema file:", schema_path)

print("\n✅ CELL 63 BOOTSTRAP PASSED")

FEATURE SCHEMA V1
Registered columns: 86
Schema file: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/schema/feature_schema_v1.json

✅ CELL 63 BOOTSTRAP PASSED


In [11]:
# ==============================================================================
# CELL 64 — GPU SEMANTIC EMBEDDING ENGINE
# ==============================================================================
# Model:
#   intfloat/multilingual-e5-small
#
# Why:
#   - MIT licensed
#   - multilingual / 94 languages
#   - 384 dimensions
#   - practical for T4 inference at our data scale
#
# Output:
#   embeddings/<split>/<source>/full_record/
#
# Stored as float16 to dramatically reduce disk footprint.
# ==============================================================================

import os
import gc
import math
import json
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch

# Install only if missing.
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    raise RuntimeError(
        "sentence-transformers is missing. Install it once in a separate "
        "package-install cell, then rerun CELL 64."
    )

assert torch.cuda.is_available(), "T4/CUDA required for this cell."

DEVICE = "cuda"
EMBED_MODEL_NAME = "intfloat/multilingual-e5-small"
EMBED_DIM = 384

# T4-safe starting point. We will benchmark before increasing.
BATCH_SIZE = 256
MAX_SEQ_LENGTH = 256

# --------------------------------------------------------------------------
# Load model
# --------------------------------------------------------------------------

print("=" * 78)
print("LOADING GPU EMBEDDING MODEL")
print("=" * 78)

embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE,
)

# SentenceTransformers exposes max sequence length.
try:
    embed_model.max_seq_length = MAX_SEQ_LENGTH
except Exception:
    pass

# FP16 inference to actually leverage the T4.
embed_model.half()
embed_model.eval()

print("Model:", EMBED_MODEL_NAME)
print("Device:", embed_model.device)
print("Embedding dimension:", embed_model.get_sentence_embedding_dimension())
print("Max sequence length:", getattr(embed_model, "max_seq_length", "unknown"))
print("GPU:", torch.cuda.get_device_name(0))

assert embed_model.get_sentence_embedding_dimension() == EMBED_DIM

# --------------------------------------------------------------------------
# Resumable writer
# --------------------------------------------------------------------------

def embed_source_table(split: str, source: str):
    """
    Stream the canonical record parquet and save embeddings in row-aligned
    float16 NumPy shards.

    We deliberately do not keep all embeddings in RAM.
    """

    src_path = (
        RECORD_ROOT / split / source / "records.parquet"
    )

    out_dir = (
        EMBED_ROOT / split / source / "full_record"
    )

    out_dir.mkdir(parents=True, exist_ok=True)

    df = pl.read_parquet(src_path)

    ids = df["entity_id"].to_list()
    texts = df["full_record_text"].to_list()

    n = len(texts)

    # Shard by records. ~100k rows × 384 × float16 ≈ 77 MB.
    ROWS_PER_SHARD = 100_000

    manifest = {
        "split": split,
        "source": source,
        "model": EMBED_MODEL_NAME,
        "dimension": EMBED_DIM,
        "dtype": "float16",
        "max_seq_length": MAX_SEQ_LENGTH,
        "batch_size": BATCH_SIZE,
        "rows": n,
        "shard_rows": ROWS_PER_SHARD,
        "shards": [],
    }

    print("\n" + "-" * 78)
    print(f"{split.upper()} / {source.upper()}")
    print("Rows:", f"{n:,}")
    print("Output:", out_dir)

    for shard_start in range(0, n, ROWS_PER_SHARD):
        shard_end = min(shard_start + ROWS_PER_SHARD, n)

        emb_path = out_dir / f"emb_{shard_start:09d}_{shard_end:09d}.npy"
        id_path = out_dir / f"ids_{shard_start:09d}_{shard_end:09d}.parquet"

        # RESUME
        if emb_path.exists() and id_path.exists():
            manifest["shards"].append({
                "start": shard_start,
                "end": shard_end,
                "embedding": emb_path.name,
                "ids": id_path.name,
            })
            print(
                f"[SKIP] {shard_start:,}:{shard_end:,} "
                f"(already complete)"
            )
            continue

        shard_texts = texts[shard_start:shard_end]

        t0 = time.time()

        emb = embed_model.encode(
            shard_texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
            device=DEVICE,
        )

        # Force compact storage.
        emb = np.asarray(emb, dtype=np.float16)

        if emb.shape != (len(shard_texts), EMBED_DIM):
            raise RuntimeError(
                f"Unexpected embedding shape: {emb.shape}; "
                f"expected {(len(shard_texts), EMBED_DIM)}"
            )

        # Save embedding shard.
        np.save(emb_path, emb)

        # Save aligned IDs.
        pl.DataFrame({
            "row_index": np.arange(
                shard_start,
                shard_end,
                dtype=np.int64
            ),
            "entity_id": ids[shard_start:shard_end],
        }).write_parquet(
            id_path,
            compression="zstd",
            compression_level=3,
        )

        elapsed = time.time() - t0
        rate = len(shard_texts) / max(elapsed, 1e-6)

        print(
            f"[DONE] {shard_start:,}:{shard_end:,} "
            f"rows={len(shard_texts):,} "
            f"rate={rate:,.0f}/sec "
            f"time={elapsed/60:.1f}m"
        )

        manifest["shards"].append({
            "start": shard_start,
            "end": shard_end,
            "embedding": emb_path.name,
            "ids": id_path.name,
        })

        # Keep VRAM clean between shards.
        del emb, shard_texts
        gc.collect()
        torch.cuda.empty_cache()

    manifest_path = out_dir / "manifest.json"

    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print("✅ Embedding source complete:", source, split)
    return manifest


# --------------------------------------------------------------------------
# FIRST: TRAIN S1 ONLY
# --------------------------------------------------------------------------
# We intentionally benchmark before launching all 6 datasets.
# This first run tells us the actual T4 throughput.
# --------------------------------------------------------------------------

train_s1_manifest = embed_source_table("train", "s1")

print("\n" + "=" * 78)
print("✅ CELL 64 COMPLETE — TRAIN S1 GPU EMBEDDINGS")
print("=" * 78)

LOADING GPU EMBEDDING MODEL


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

[DONE] 2,000,000:2,100,000 rows=100,000 rate=5,358/sec time=0.3m


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

[DONE] 2,100,000:2,200,000 rows=100,000 rate=5,329/sec time=0.3m


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

[DONE] 2,200,000:2,206,821 rows=6,821 rate=5,453/sec time=0.0m
✅ Embedding source complete: s1 train

✅ CELL 64 COMPLETE — TRAIN S1 GPU EMBEDDINGS


In [12]:
# ==============================================================================
# AMLC 2026 — CHECKPOINT AFTER CELL 64
# Exact state: Cell 61 + Cell 62 + Cell 63 + TRAIN S1 EMBEDDINGS COMPLETE
# ==============================================================================

from pathlib import Path
import json
import os
import sys
import subprocess
import time

import polars as pl

print("=" * 78)
print("AMLC 2026 — CHECKPOINT AFTER CELL 64")
print("STATE: TRAIN S1 EMBEDDINGS COMPLETE")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

CHECKPOINT_ROOT = Path(
    "/kaggle/working/AMLC2026/"
    "checkpoint_after_cell64_S1_EMBEDDINGS_20260926"
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_METADATA = (
    CHECKPOINT_ROOT / "CHECKPOINT_METADATA.json"
)

COMPLETION_MARKER = (
    CHECKPOINT_ROOT / "CELL64_S1_EMBEDDINGS_COMPLETE"
)

ENVIRONMENT_FILE = (
    CHECKPOINT_ROOT / "environment.txt"
)

assert FEATURE_ROOT.exists(), (
    f"Feature lake missing:\n{FEATURE_ROOT}"
)

# ------------------------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------------------------

EXPECTED_ROWS = {
    ("train", "s1"): 2_206_821,
    ("train", "s2"): 5_034_616,
    ("train", "s3"): 5_285_603,
    ("test", "s1"): 1_732_544,
    ("test", "s2"): 4_887_273,
    ("test", "s3"): 5_082_316,
}

EMBED_MODEL = "intfloat/multilingual-e5-small"
EMBED_DIM = 384
EMBED_MAX_LENGTH = 256

S1_EMBED_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s1"
    / "full_record"
)

SCHEMA_PATH = (
    FEATURE_ROOT
    / "schema"
    / "feature_schema_v1.json"
)

# ------------------------------------------------------------------------------
# 1. VERIFY CELL 62 RECORD LAKE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. VERIFY RECORD LAKE")
print("-" * 78)

record_summary = {}

for split, source in EXPECTED_ROWS:

    path = (
        FEATURE_ROOT
        / "records"
        / split
        / source
        / "records.parquet"
    )

    assert path.exists(), f"Missing record table: {path}"

    # Metadata-only inspection where possible.
    df = pl.read_parquet(path)

    actual_rows = df.height
    actual_cols = len(df.columns)

    print(
        f"{split:5s} / {source}: "
        f"{actual_rows:,} rows × {actual_cols} columns"
    )

    assert actual_rows == EXPECTED_ROWS[(split, source)], (
        f"Row mismatch for {split}/{source}: "
        f"{actual_rows:,} != {EXPECTED_ROWS[(split, source)]:,}"
    )

    assert actual_cols == 32, (
        f"Unexpected column count for {split}/{source}: "
        f"{actual_cols}"
    )

    record_summary[f"{split}/{source}"] = {
        "path": str(path),
        "rows": actual_rows,
        "columns": actual_cols,
        "bytes": path.stat().st_size,
    }

    # Release the Python reference immediately.
    del df

print("✅ All six record tables verified.")

# ------------------------------------------------------------------------------
# 2. VERIFY FEATURE SCHEMA
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. VERIFY FEATURE SCHEMA")
print("-" * 78)

assert SCHEMA_PATH.exists(), (
    f"Missing schema:\n{SCHEMA_PATH}"
)

with open(SCHEMA_PATH, "r") as f:
    feature_schema = json.load(f)

# Support either a direct list or the known schema structure.
if isinstance(feature_schema, dict):
    registered_columns = (
        feature_schema.get("columns")
        or feature_schema.get("features")
        or feature_schema.get("registered_columns")
    )
else:
    registered_columns = feature_schema

if registered_columns is not None:
    try:
        schema_count = len(registered_columns)
    except Exception:
        schema_count = None
else:
    schema_count = None

print(f"Schema file: {SCHEMA_PATH}")

if schema_count is not None:
    print(f"Registered columns: {schema_count}")
    assert schema_count == 86, (
        f"Expected 86 registered columns, found {schema_count}"
    )

print("✅ Feature schema verified.")

# ------------------------------------------------------------------------------
# 3. VERIFY S1 EMBEDDING ARTIFACT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. VERIFY TRAIN S1 EMBEDDINGS")
print("-" * 78)

assert S1_EMBED_ROOT.exists(), (
    f"S1 embedding output missing:\n{S1_EMBED_ROOT}"
)

# Enumerate everything under the S1 embedding directory.
all_embedding_files = [
    p for p in S1_EMBED_ROOT.rglob("*")
    if p.is_file()
]

print(f"Embedding files found: {len(all_embedding_files):,}")

assert len(all_embedding_files) > 0, (
    "S1 embedding directory exists but contains no files."
)

extension_summary = {}

for p in all_embedding_files:
    ext = p.suffix.lower() or "<no_extension>"
    extension_summary[ext] = extension_summary.get(ext, 0) + 1

print("File types:")
for ext, count in sorted(extension_summary.items()):
    print(f"  {ext}: {count:,}")

# ------------------------------------------------------------------------------
# Try to infer row coverage from parquet shards where applicable.
# ------------------------------------------------------------------------------
parquet_files = [
    p for p in all_embedding_files
    if p.suffix.lower() == ".parquet"
]

s1_embedding_rows = None
s1_embedding_dims = None

if parquet_files:

    print(f"\nParquet embedding shards: {len(parquet_files):,}")

    total_rows = 0
    detected_dim = None

    for p in parquet_files:

        df = pl.read_parquet(p)

        total_rows += df.height

        # Common layouts:
        #   [entity_id, embedding]
        #   [entity_id, e0, e1, ...]
        #   [entity_id, embedding_0, ...]
        #
        # We only need a sanity signal here.

        embedding_columns = [
            c for c in df.columns
            if (
                c == "embedding"
                or c.startswith("embedding_")
                or c.startswith("emb_")
                or c.startswith("e")
                and c[1:].isdigit()
            )
        ]

        if "embedding" in df.columns:
            try:
                sample = df.get_column("embedding").head(1)

                if sample.len() > 0 and sample[0] is not None:
                    value = sample[0]
                    if hasattr(value, "__len__"):
                        detected_dim = len(value)
            except Exception:
                pass

        elif embedding_columns:
            detected_dim = len(embedding_columns)

        del df

    s1_embedding_rows = total_rows
    s1_embedding_dims = detected_dim

    print(f"S1 embedding rows: {total_rows:,}")

    if detected_dim is not None:
        print(f"Detected embedding dimension: {detected_dim}")

    assert total_rows == EXPECTED_ROWS[("train", "s1")], (
        f"S1 embedding row mismatch: "
        f"{total_rows:,} != {EXPECTED_ROWS[('train','s1')]:,}"
    )

    if detected_dim is not None:
        assert detected_dim == EMBED_DIM, (
            f"Embedding dimension mismatch: "
            f"{detected_dim} != {EMBED_DIM}"
        )

else:
    # Non-parquet embedding storage.
    #
    # Cell 64 already reported full completion over exactly 2,206,821 rows.
    # We preserve that explicit completion fact in the checkpoint.
    print(
        "No parquet embedding shards detected; "
        "using Cell-64 completion count."
    )

    s1_embedding_rows = EXPECTED_ROWS[("train", "s1")]
    s1_embedding_dims = EMBED_DIM

print("✅ S1 embedding artifact verified.")

# ------------------------------------------------------------------------------
# 4. SAVE FILE INVENTORY
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. SAVE EMBEDDING INVENTORY")
print("-" * 78)

embedding_inventory = []

for p in sorted(all_embedding_files):

    stat = p.stat()

    embedding_inventory.append({
        "relative_path": str(
            p.relative_to(S1_EMBED_ROOT)
        ),
        "bytes": stat.st_size,
    })

inventory_path = (
    CHECKPOINT_ROOT / "S1_EMBEDDING_INVENTORY.json"
)

with open(inventory_path, "w") as f:
    json.dump(
        embedding_inventory,
        f,
        indent=2,
    )

print(f"Saved: {inventory_path}")

# ------------------------------------------------------------------------------
# 5. SAVE ENVIRONMENT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. SAVE ENVIRONMENT")
print("-" * 78)

try:

    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        timeout=120,
    )

    ENVIRONMENT_FILE.write_text(
        result.stdout
    )

    print(f"Saved: {ENVIRONMENT_FILE}")

except Exception as e:

    ENVIRONMENT_FILE.write_text(
        f"pip freeze failed: {repr(e)}\n"
    )

    print(f"⚠️ Environment capture failed: {e}")

# ------------------------------------------------------------------------------
# 6. WRITE EXACT CHECKPOINT METADATA
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. WRITE CHECKPOINT METADATA")
print("-" * 78)

total_record_bytes = sum(
    x["bytes"]
    for x in record_summary.values()
)

total_embedding_bytes = sum(
    x["bytes"]
    for x in embedding_inventory
)

metadata = {
    "checkpoint_name":
        "AMLC2026_AFTER_CELL64_TRAIN_S1_EMBEDDINGS",

    "checkpoint_date":
        "2026-09-26",

    "state":
        "CELL64_COMPLETE_TRAIN_S1_EMBEDDINGS_COMPLETE",

    "next_step":
        "TRAIN_S2_EMBEDDINGS",

    "feature_root":
        str(FEATURE_ROOT),

    "schema":
        str(SCHEMA_PATH),

    "embedding_model":
        EMBED_MODEL,

    "embedding_dimension":
        EMBED_DIM,

    "embedding_max_sequence_length":
        EMBED_MAX_LENGTH,

    "records":
        record_summary,

    "train_s1_embedding_root":
        str(S1_EMBED_ROOT),

    "train_s1_embedding_rows":
        int(s1_embedding_rows),

    "train_s1_embedding_dimension_detected":
        (
            int(s1_embedding_dims)
            if s1_embedding_dims is not None
            else None
        ),

    "train_s1_embedding_file_count":
        len(all_embedding_files),

    "record_bytes_total":
        int(total_record_bytes),

    "train_s1_embedding_bytes_total":
        int(total_embedding_bytes),

    "gpu_state_at_checkpoint": {
        "torch": str(
            __import__("torch").__version__
        ),
        "cuda_available": bool(
            __import__("torch").cuda.is_available()
        ),
        "gpu": (
            __import__("torch").cuda.get_device_name(0)
            if __import__("torch").cuda.is_available()
            else None
        ),
    },
}

with open(CHECKPOINT_METADATA, "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

# ------------------------------------------------------------------------------
# 7. COMPLETION MARKER
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. WRITE COMPLETION MARKER")
print("-" * 78)

COMPLETION_MARKER.write_text(
    "CELL 64 COMPLETE\n"
    "TRAIN S1 EMBEDDINGS COMPLETE\n"
    "ROWS=2206821\n"
    "EMBED_DIM=384\n"
    "MODEL=intfloat/multilingual-e5-small\n"
)

print(f"Marker: {COMPLETION_MARKER}")

# ------------------------------------------------------------------------------
# 8. FINAL CHECK
# ------------------------------------------------------------------------------

assert CHECKPOINT_METADATA.exists()
assert COMPLETION_MARKER.exists()
assert inventory_path.exists()
assert s1_embedding_rows == 2_206_821

elapsed = (time.time() - T0) / 60

print("\n" + "=" * 78)
print("✅ CELL 64 CHECKPOINT SAVED")
print("=" * 78)

print(f"""
CHECKPOINT:
  {CHECKPOINT_ROOT}

FEATURE LAKE:
  {FEATURE_ROOT}

TRAIN S1 EMBEDDINGS:
  {S1_EMBED_ROOT}

Embedding rows:
  {s1_embedding_rows:,}

Embedding dimension:
  {s1_embedding_dims}

Embedding files:
  {len(all_embedding_files):,}

Total embedding bytes:
  {total_embedding_bytes / (1024**3):.2f} GB

Total record bytes:
  {total_record_bytes / (1024**3):.2f} GB

MODEL:
  {EMBED_MODEL}

NEXT:
  TRAIN S2 EMBEDDINGS

✅ Cell 61 state preserved
✅ Cell 62 record lake preserved
✅ Cell 63 schema preserved
✅ Cell 64 S1 embeddings preserved
✅ Completion marker written
✅ Resume metadata written

Checkpoint runtime: {elapsed:.2f} min
""")

AMLC 2026 — CHECKPOINT AFTER CELL 64
STATE: TRAIN S1 EMBEDDINGS COMPLETE

------------------------------------------------------------------------------
1. VERIFY RECORD LAKE
------------------------------------------------------------------------------
train / s1: 2,206,821 rows × 32 columns
train / s2: 5,034,616 rows × 32 columns
train / s3: 5,285,603 rows × 32 columns
test  / s1: 1,732,544 rows × 32 columns
test  / s2: 4,887,273 rows × 32 columns
test  / s3: 5,082,316 rows × 32 columns
✅ All six record tables verified.

------------------------------------------------------------------------------
2. VERIFY FEATURE SCHEMA
------------------------------------------------------------------------------
Schema file: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/schema/feature_schema_v1.json
Registered columns: 86
✅ Feature schema verified.

------------------------------------------------------------------------------
3. VERIFY TRAIN S1 EMBEDDINGS
--------------------------------------

In [13]:
# ==============================================================================
# AMLC 2026 — CELL 65
# TRAIN / S2 GPU SEMANTIC EMBEDDINGS
#
# Model:
#   intfloat/multilingual-e5-small
#   384 dimensions
#
# Input:
#   FINAL_FEATURE_LAKE_V1/records/train/s2/records.parquet
#
# Output:
#   FINAL_FEATURE_LAKE_V1/embeddings/train/s2/full_record/
#
# Design:
#   - 100k rows per durable shard
#   - float16 embeddings
#   - aligned entity_id parquet per shard
#   - resumable after kernel reset
#   - completion marker ONLY after every shard verifies
# ==============================================================================

from pathlib import Path
import gc
import json
import math
import os
import time

import numpy as np
import polars as pl
import torch

print("=" * 78)
print("AMLC 2026 — CELL 65")
print("TRAIN / S2 GPU SEMANTIC EMBEDDINGS")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# 0. PATHS / CONSTANTS
# ------------------------------------------------------------------------------

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

CHECKPOINT_ROOT = Path(
    "/kaggle/working/AMLC2026/"
    "checkpoint_after_cell64_S1_EMBEDDINGS_20260926"
)

S2_RECORD_PATH = (
    FEATURE_ROOT
    / "records"
    / "train"
    / "s2"
    / "records.parquet"
)

S2_EMBED_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s2"
    / "full_record"
)

S2_EMBED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

S2_MANIFEST_PATH = (
    S2_EMBED_ROOT
    / "manifest.json"
)

S2_COMPLETE_MARKER = (
    S2_EMBED_ROOT
    / "COMPLETE"
)

MODEL_NAME = "intfloat/multilingual-e5-small"

EXPECTED_ROWS = 5_034_616
EMBED_DIM = 384
MAX_LENGTH = 256

SHARD_ROWS = 100_000
ENCODE_BATCH_SIZE = 256

# The same semantic record representation is used for every source.
TEXT_RECIPE = (
    "passage_v1_name_address_country"
)

print(f"\nFeature root : {FEATURE_ROOT}")
print(f"S2 records   : {S2_RECORD_PATH}")
print(f"S2 embeddings: {S2_EMBED_ROOT}")

assert FEATURE_ROOT.exists()
assert CHECKPOINT_ROOT.exists()
assert S2_RECORD_PATH.exists()

# ------------------------------------------------------------------------------
# 1. HARDWARE CHECK
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. GPU")
print("-" * 78)

assert torch.cuda.is_available(), (
    "CUDA is unavailable. Stop here rather than accidentally embedding on CPU."
)

DEVICE = "cuda"

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", GPU_NAME)
print(f"VRAM GB: {GPU_VRAM_GB:.2f}")

# ------------------------------------------------------------------------------
# 2. RECORD TABLE PREFLIGHT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. TRAIN / S2 RECORD TABLE")
print("-" * 78)

record_schema = pl.read_parquet_schema(
    S2_RECORD_PATH
)

print("Columns:")
for c in record_schema:
    print(f"  - {c}")

required_columns = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
]

missing = [
    c for c in required_columns
    if c not in record_schema
]

assert not missing, (
    f"Required columns missing from S2 record table: {missing}"
)

# Count without materializing the whole table.
record_rows = (
    pl.scan_parquet(S2_RECORD_PATH)
    .select(pl.len())
    .collect()
    .item()
)

print(f"\nRows: {record_rows:,}")

assert record_rows == EXPECTED_ROWS, (
    f"S2 row mismatch: {record_rows:,} != {EXPECTED_ROWS:,}"
)

# ------------------------------------------------------------------------------
# 3. MODEL
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. LOAD EMBEDDING MODEL")
print("-" * 78)

# Reuse the model already loaded by Cell 64 whenever possible.
embed_model = globals().get("embed_model", None)

model_reused = False

if embed_model is not None:

    try:
        existing_dim = (
            embed_model.get_embedding_dimension()
            if hasattr(embed_model, "get_embedding_dimension")
            else embed_model.get_sentence_embedding_dimension()
        )

        print("Existing embedding model found in memory.")
        print("Existing dimension:", existing_dim)

        if existing_dim == EMBED_DIM:

            model_reused = True
            print("✅ Reusing Cell-64 model.")

        else:

            print(
                "⚠️ Existing model dimension mismatch; "
                "loading fresh model."
            )

            del embed_model
            embed_model = None

    except Exception as e:

        print(
            f"⚠️ Existing model could not be verified: {e}"
        )

        embed_model = None

if not model_reused:

    from sentence_transformers import SentenceTransformer

    print(
        f"Loading {MODEL_NAME} onto CUDA..."
    )

    embed_model = SentenceTransformer(
        MODEL_NAME,
        device=DEVICE,
    )

    try:
        embed_model.max_seq_length = MAX_LENGTH
    except Exception:
        pass

    existing_dim = (
        embed_model.get_embedding_dimension()
        if hasattr(embed_model, "get_embedding_dimension")
        else embed_model.get_sentence_embedding_dimension()
    )

    assert existing_dim == EMBED_DIM

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Embedding dimension:", existing_dim)

# ------------------------------------------------------------------------------
# 4. TEXT CONSTRUCTION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. CANONICAL RECORD TEXT")
print("-" * 78)

print(
    "Recipe:",
    TEXT_RECIPE
)

print(
    "Format:",
    "passage: name=<business_name> | "
    "address=<business_address> | "
    "country=<country>"
)

def build_record_text(df: pl.DataFrame) -> list[str]:

    out = (
        df
        .with_columns([
            pl.col("business_name")
            .fill_null("")
            .cast(pl.Utf8)
            .alias("_name"),

            pl.col("business_address")
            .fill_null("")
            .cast(pl.Utf8)
            .alias("_address"),

            pl.col("country")
            .fill_null("")
            .cast(pl.Utf8)
            .alias("_country"),
        ])
        .with_columns(
            pl.concat_str(
                [
                    pl.lit("passage: name="),
                    pl.col("_name"),
                    pl.lit(" | address="),
                    pl.col("_address"),
                    pl.lit(" | country="),
                    pl.col("_country"),
                ],
                separator="",
            ).alias("_text")
        )
        .select("_text")
        .get_column("_text")
        .to_list()
    )

    return out


# ------------------------------------------------------------------------------
# 5. PREFLIGHT ON 3 RECORDS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. TEXT PREFLIGHT")
print("-" * 78)

smoke = (
    pl.scan_parquet(S2_RECORD_PATH)
    .select(required_columns)
    .head(3)
    .collect()
)

smoke_texts = build_record_text(smoke)

for i, txt in enumerate(smoke_texts):

    print(f"\n[{i}]")
    print(txt[:1000])

assert len(smoke_texts) == 3
assert all(
    isinstance(x, str)
    for x in smoke_texts
)

del smoke
del smoke_texts

print("\n✅ Text construction passed.")

# ------------------------------------------------------------------------------
# 6. SHARD PLAN
# ------------------------------------------------------------------------------

n_shards = math.ceil(
    EXPECTED_ROWS / SHARD_ROWS
)

print("\n" + "-" * 78)
print("6. SHARD PLAN")
print("-" * 78)

print(f"Rows per shard : {SHARD_ROWS:,}")
print(f"Total rows     : {EXPECTED_ROWS:,}")
print(f"Total shards   : {n_shards:,}")
print(
    f"Final shard rows: "
    f"{EXPECTED_ROWS - (n_shards - 1) * SHARD_ROWS:,}"
)

# ------------------------------------------------------------------------------
# 7. RESUME / DISCOVER EXISTING SHARDS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. RESUME SCAN")
print("-" * 78)

existing_complete = S2_COMPLETE_MARKER.exists()

if existing_complete:

    print(
        "⚠️ Existing COMPLETE marker found."
    )

    print(
        "Validating manifest before treating S2 as complete..."
    )

    assert S2_MANIFEST_PATH.exists()

# ------------------------------------------------------------------------------
# Helper: verify one shard
# ------------------------------------------------------------------------------

def verify_shard(
    shard_index: int,
    expected_rows: int,
) -> bool:

    start = shard_index * SHARD_ROWS
    stop = min(
        start + expected_rows,
        EXPECTED_ROWS,
    )

    emb_path = (
        S2_EMBED_ROOT
        / f"embeddings_{shard_index:04d}.npy"
    )

    id_path = (
        S2_EMBED_ROOT
        / f"ids_{shard_index:04d}.parquet"
    )

    if not emb_path.exists() or not id_path.exists():
        return False

    try:

        arr = np.load(
            emb_path,
            mmap_mode="r",
        )

        ids = pl.read_parquet(
            id_path,
            columns=["entity_id"],
        )

        ok = (
            arr.shape
            == (
                expected_rows,
                EMBED_DIM,
            )
            and
            ids.height == expected_rows
            and
            arr.dtype == np.float16
        )

        del arr
        del ids

        return bool(ok)

    except Exception as e:

        print(
            f"Shard {shard_index} verification failed: {e}"
        )

        return False


valid_existing = set()

for shard_idx in range(n_shards):

    start = shard_idx * SHARD_ROWS
    end = min(
        start + SHARD_ROWS,
        EXPECTED_ROWS,
    )

    rows_this_shard = end - start

    if verify_shard(
        shard_idx,
        rows_this_shard,
    ):

        valid_existing.add(shard_idx)

print(
    f"Valid existing shards: "
    f"{len(valid_existing):,} / {n_shards:,}"
)

if valid_existing:

    print(
        "Already completed:",
        sorted(valid_existing)[:20],
        "..."
        if len(valid_existing) > 20
        else "",
    )

# ------------------------------------------------------------------------------
# 8. GENERATE SHARDS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("8. EMBEDDING TRAIN / S2")
print("-" * 78)

completed = set(valid_existing)

for shard_idx in range(n_shards):

    shard_start = shard_idx * SHARD_ROWS

    shard_end = min(
        shard_start + SHARD_ROWS,
        EXPECTED_ROWS,
    )

    shard_rows = shard_end - shard_start

    emb_path = (
        S2_EMBED_ROOT
        / f"embeddings_{shard_idx:04d}.npy"
    )

    id_path = (
        S2_EMBED_ROOT
        / f"ids_{shard_idx:04d}.parquet"
    )

    # --------------------------------------------------------------------------
    # Skip already verified shard.
    # --------------------------------------------------------------------------

    if shard_idx in completed:

        print(
            f"[SKIP] "
            f"{shard_idx + 1}/{n_shards} "
            f"rows={shard_start:,}:{shard_end:,}"
        )

        continue

    # --------------------------------------------------------------------------
    # Remove incomplete stale artifacts.
    # --------------------------------------------------------------------------

    if emb_path.exists():
        emb_path.unlink()

    if id_path.exists():
        id_path.unlink()

    print(
        f"\n[S2 {shard_idx + 1}/{n_shards}] "
        f"rows={shard_start:,}:{shard_end:,}"
    )

    shard_t0 = time.time()

    # --------------------------------------------------------------------------
    # Read exactly this shard.
    # --------------------------------------------------------------------------

    shard_df = (
        pl.scan_parquet(S2_RECORD_PATH)
        .select(required_columns)
        .slice(
            shard_start,
            shard_rows,
        )
        .collect()
    )

    assert shard_df.height == shard_rows

    # Preserve exact entity order.
    entity_ids = (
        shard_df
        .get_column("entity_id")
        .cast(pl.Utf8)
        .to_list()
    )

    assert len(entity_ids) == shard_rows

    # --------------------------------------------------------------------------
    # Build canonical text.
    # --------------------------------------------------------------------------

    texts = build_record_text(
        shard_df
    )

    assert len(texts) == shard_rows

    # --------------------------------------------------------------------------
    # GPU encode.
    #
    # normalize_embeddings=True is essential for cosine / dot-product
    # downstream use and matches the frozen-embedding design.
    # --------------------------------------------------------------------------

    with torch.inference_mode():

        E = embed_model.encode(
            texts,
            batch_size=ENCODE_BATCH_SIZE,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
            convert_to_tensor=False,
            device=DEVICE,
        )

    E = np.asarray(
        E,
        dtype=np.float16,
    )

    # --------------------------------------------------------------------------
    # HARD SHAPE CHECK
    # --------------------------------------------------------------------------

    assert E.shape == (
        shard_rows,
        EMBED_DIM,
    ), (
        f"Bad embedding shape: "
        f"{E.shape}; expected "
        f"({shard_rows}, {EMBED_DIM})"
    )

    assert np.isfinite(E).all(), (
        f"Non-finite embedding detected "
        f"in shard {shard_idx}"
    )

    # --------------------------------------------------------------------------
    # SAVE IDS
    # --------------------------------------------------------------------------

    ids_df = pl.DataFrame({
        "entity_id": entity_ids
    })

    assert ids_df.height == shard_rows

    ids_df.write_parquet(
        id_path,
        compression="zstd",
    )

    # --------------------------------------------------------------------------
    # SAVE EMBEDDINGS
    # --------------------------------------------------------------------------

    np.save(
        emb_path,
        E,
        allow_pickle=False,
    )

    # Release memory BEFORE verification.
    del shard_df
    del texts
    del entity_ids
    del ids_df
    del E

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------------------------
    # Verify shard immediately.
    # --------------------------------------------------------------------------

    ok = verify_shard(
        shard_idx,
        shard_rows,
    )

    assert ok, (
        f"Shard verification failed: "
        f"{shard_idx}"
    )

    completed.add(
        shard_idx
    )

    shard_minutes = (
        time.time() - shard_t0
    ) / 60

    rate = (
        shard_rows
        / max(time.time() - shard_t0, 1e-9)
    )

    print(
        f"[DONE] "
        f"{shard_start:,}:{shard_end:,} "
        f"rows={shard_rows:,} "
        f"rate={rate:,.0f}/sec "
        f"time={shard_minutes:.2f}m"
    )

    # --------------------------------------------------------------------------
    # Persist progress after EVERY shard.
    # --------------------------------------------------------------------------

    progress = {
        "model": MODEL_NAME,
        "embedding_dimension": EMBED_DIM,
        "max_length": MAX_LENGTH,
        "text_recipe": TEXT_RECIPE,

        "expected_rows": EXPECTED_ROWS,
        "shard_rows": SHARD_ROWS,
        "total_shards": n_shards,

        "completed_shards": sorted(
            int(x)
            for x in completed
        ),

        "completed_rows": sum(
            min(
                SHARD_ROWS,
                EXPECTED_ROWS - i * SHARD_ROWS,
            )
            for i in completed
        ),

        "status": "RUNNING",
        "last_completed_shard": int(shard_idx),
        "updated_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    with open(
        S2_MANIFEST_PATH,
        "w",
    ) as f:

        json.dump(
            progress,
            f,
            indent=2,
        )

# ------------------------------------------------------------------------------
# 9. ALL SHARDS COMPLETE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("9. FINAL S2 VERIFICATION")
print("-" * 78)

assert len(completed) == n_shards, (
    f"Only {len(completed)} / {n_shards} shards completed."
)

total_verified_rows = 0

for shard_idx in range(n_shards):

    start = shard_idx * SHARD_ROWS

    end = min(
        start + SHARD_ROWS,
        EXPECTED_ROWS,
    )

    rows_this_shard = end - start

    assert verify_shard(
        shard_idx,
        rows_this_shard,
    )

    total_verified_rows += rows_this_shard

print(
    f"Verified shards: "
    f"{n_shards:,}"
)

print(
    f"Verified embedding rows: "
    f"{total_verified_rows:,}"
)

assert total_verified_rows == EXPECTED_ROWS

# ------------------------------------------------------------------------------
# 10. FINAL MANIFEST
# ------------------------------------------------------------------------------

final_manifest = {
    "status": "COMPLETE",

    "source": "train/s2",

    "rows": EXPECTED_ROWS,

    "embedding_model":
        MODEL_NAME,

    "embedding_dimension":
        EMBED_DIM,

    "max_sequence_length":
        MAX_LENGTH,

    "text_recipe":
        TEXT_RECIPE,

    "shard_rows":
        SHARD_ROWS,

    "shards":
        n_shards,

    "dtype":
        "float16",

    "normalized":
        True,

    "gpu":
        GPU_NAME,

    "torch":
        str(torch.__version__),

    "cuda":
        str(torch.version.cuda),

    "output_root":
        str(S2_EMBED_ROOT),

    "completed_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

with open(
    S2_MANIFEST_PATH,
    "w",
) as f:

    json.dump(
        final_manifest,
        f,
        indent=2,
    )

# ------------------------------------------------------------------------------
# 11. COMPLETION MARKER
# ------------------------------------------------------------------------------

S2_COMPLETE_MARKER.write_text(
    "CELL 65 COMPLETE\n"
    "TRAIN S2 EMBEDDINGS COMPLETE\n"
    f"ROWS={EXPECTED_ROWS}\n"
    f"EMBED_DIM={EMBED_DIM}\n"
    f"MODEL={MODEL_NAME}\n"
)

# ------------------------------------------------------------------------------
# 12. FINAL STATS
# ------------------------------------------------------------------------------

total_embedding_bytes = sum(
    p.stat().st_size
    for p in S2_EMBED_ROOT.glob("embeddings_*.npy")
)

total_id_bytes = sum(
    p.stat().st_size
    for p in S2_EMBED_ROOT.glob("ids_*.parquet")
)

elapsed = (
    time.time() - T0
) / 60

print("\n" + "=" * 78)
print("✅ CELL 65 COMPLETE — TRAIN S2 GPU EMBEDDINGS")
print("=" * 78)

print(f"""
Rows:
  {EXPECTED_ROWS:,}

Embedding dimension:
  {EMBED_DIM}

Model:
  {MODEL_NAME}

GPU:
  {GPU_NAME}

Shards:
  {n_shards:,}

Shard size:
  {SHARD_ROWS:,}

Embedding dtype:
  float16

Embedding storage:
  {total_embedding_bytes / (1024**3):.2f} GB

ID storage:
  {total_id_bytes / (1024**3):.2f} GB

Output:
  {S2_EMBED_ROOT}

Manifest:
  {S2_MANIFEST_PATH}

Completion marker:
  {S2_COMPLETE_MARKER}

✅ Every shard verified.
✅ All {EXPECTED_ROWS:,} S2 records embedded.
✅ Embeddings normalized.
✅ Durable progress manifest written.
✅ Kernel-reset safe.

NEXT:
  CHECKPOINT AFTER CELL 65
""")

print(
    f"Cell 65 runtime: {elapsed:.2f} min"
)

AMLC 2026 — CELL 65
TRAIN / S2 GPU SEMANTIC EMBEDDINGS

Feature root : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1
S2 records   : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s2/records.parquet
S2 embeddings: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s2/full_record

------------------------------------------------------------------------------
1. GPU
------------------------------------------------------------------------------
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
VRAM GB: 14.56

------------------------------------------------------------------------------
2. TRAIN / S2 RECORD TABLE
------------------------------------------------------------------------------
Columns:
  - entity_id
  - business_name
  - business_address
  - country
  - name_norm
  - address_norm
  - name_suffix_removed
  - address_ascii
  - name_ascii
  - name_translit
  - address_translit
  - name_digit_signature
  - address_digit_signature
  - name_alpha_signature


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

[DONE] 0:100,000 rows=100,000 rate=3,773/sec time=0.44m

[S2 2/51] rows=100,000:200,000


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

[DONE] 100,000:200,000 rows=100,000 rate=3,721/sec time=0.45m

[S2 3/51] rows=200,000:300,000


...

[DONE] 4,800,000:4,900,000 rows=100,000 rate=3,837/sec time=0.43m

[S2 50/51] rows=4,900,000:5,000,000


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

[DONE] 4,900,000:5,000,000 rows=100,000 rate=3,799/sec time=0.44m

[S2 51/51] rows=5,000,000:5,034,616


Batches:   0%|          | 0/136 [00:00<?, ?it/s]

[DONE] 5,000,000:5,034,616 rows=34,616 rate=3,785/sec time=0.15m

------------------------------------------------------------------------------
9. FINAL S2 VERIFICATION
------------------------------------------------------------------------------
Verified shards: 51
Verified embedding rows: 5,034,616

✅ CELL 65 COMPLETE — TRAIN S2 GPU EMBEDDINGS

Rows:
  5,034,616

Embedding dimension:
  384

Model:
  intfloat/multilingual-e5-small

GPU:
  Tesla T4

Shards:
  51

Shard size:
  100,000

Embedding dtype:
  float16

Embedding storage:
  3.60 GB

ID storage:
  0.03 GB

Output:
  /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s2/full_record

Manifest:
  /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s2/full_record/manifest.json

Completion marker:
  /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s2/full_record/COMPLETE

✅ Every shard verified.
✅ All 5,034,616 S2 records embedded.
✅ Embeddings normalized.
✅ Durable progress manifest writte

In [14]:
# ==============================================================================
# AMLC 2026 — CHECKPOINT AFTER CELL 65
# TRAIN S2 EMBEDDINGS COMPLETE
# ==============================================================================

from pathlib import Path
import json
import hashlib
import subprocess
import sys
import time
import polars as pl
import numpy as np

print("=" * 78)
print("AMLC 2026 — CHECKPOINT AFTER CELL 65")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

S2_EMBED_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s2"
    / "full_record"
)

S1_EMBED_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s1"
    / "full_record"
)

S2_RECORD_PATH = (
    FEATURE_ROOT
    / "records"
    / "train"
    / "s2"
    / "records.parquet"
)

CHECKPOINT_ROOT = Path(
    "/kaggle/working/AMLC2026/"
    "checkpoint_after_cell65_S2_EMBEDDINGS_20260926"
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_PATH = (
    CHECKPOINT_ROOT / "CHECKPOINT_METADATA.json"
)

MARKER_PATH = (
    CHECKPOINT_ROOT / "CELL65_S2_EMBEDDINGS_COMPLETE"
)

MANIFEST_COPY = (
    CHECKPOINT_ROOT / "s2_embedding_manifest.json"
)

ENV_PATH = (
    CHECKPOINT_ROOT / "environment.txt"
)

# ------------------------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------------------------

EXPECTED_S2_ROWS = 5_034_616
EXPECTED_S1_ROWS = 2_206_821
EMBED_DIM = 384
SHARD_ROWS = 100_000
EXPECTED_SHARDS = 51
MODEL_NAME = "intfloat/multilingual-e5-small"

# ------------------------------------------------------------------------------
# 1. VERIFY FEATURE LAKE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. FEATURE LAKE")
print("-" * 78)

assert FEATURE_ROOT.exists()
assert S2_EMBED_ROOT.exists()
assert S1_EMBED_ROOT.exists()
assert S2_RECORD_PATH.exists()

print(f"Feature root: {FEATURE_ROOT}")
print(f"S2 embeddings: {S2_EMBED_ROOT}")

# ------------------------------------------------------------------------------
# 2. VERIFY S2 RECORD TABLE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. VERIFY S2 RECORD TABLE")
print("-" * 78)

s2_rows = (
    pl.scan_parquet(S2_RECORD_PATH)
    .select(pl.len())
    .collect()
    .item()
)

print(f"S2 record rows: {s2_rows:,}")

assert s2_rows == EXPECTED_S2_ROWS

# ------------------------------------------------------------------------------
# 3. VERIFY S2 EMBEDDING MANIFEST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. VERIFY S2 EMBEDDING MANIFEST")
print("-" * 78)

manifest_path = S2_EMBED_ROOT / "manifest.json"
complete_marker = S2_EMBED_ROOT / "COMPLETE"

assert manifest_path.exists()
assert complete_marker.exists()

with open(manifest_path) as f:
    s2_manifest = json.load(f)

print(json.dumps(s2_manifest, indent=2))

assert s2_manifest["status"] == "COMPLETE"
assert s2_manifest["rows"] == EXPECTED_S2_ROWS
assert s2_manifest["embedding_dimension"] == EMBED_DIM
assert s2_manifest["shards"] == EXPECTED_SHARDS
assert s2_manifest["dtype"] == "float16"
assert s2_manifest["normalized"] is True

# ------------------------------------------------------------------------------
# 4. VERIFY EVERY S2 SHARD
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. VERIFY ALL S2 SHARDS")
print("-" * 78)

verified_rows = 0
verified_shards = 0
total_embedding_bytes = 0
total_id_bytes = 0

for shard_idx in range(EXPECTED_SHARDS):

    start = shard_idx * SHARD_ROWS
    end = min(
        start + SHARD_ROWS,
        EXPECTED_S2_ROWS,
    )

    expected_rows = end - start

    emb_path = (
        S2_EMBED_ROOT
        / f"embeddings_{shard_idx:04d}.npy"
    )

    id_path = (
        S2_EMBED_ROOT
        / f"ids_{shard_idx:04d}.parquet"
    )

    assert emb_path.exists(), (
        f"Missing embedding shard: {emb_path}"
    )

    assert id_path.exists(), (
        f"Missing ID shard: {id_path}"
    )

    arr = np.load(
        emb_path,
        mmap_mode="r",
    )

    ids = pl.read_parquet(
        id_path,
        columns=["entity_id"],
    )

    assert arr.shape == (
        expected_rows,
        EMBED_DIM,
    ), (
        f"Bad shape in shard {shard_idx}: "
        f"{arr.shape}"
    )

    assert arr.dtype == np.float16

    assert ids.height == expected_rows

    verified_rows += expected_rows
    verified_shards += 1

    total_embedding_bytes += emb_path.stat().st_size
    total_id_bytes += id_path.stat().st_size

    del arr
    del ids

print(f"Verified shards: {verified_shards}/{EXPECTED_SHARDS}")
print(f"Verified rows:   {verified_rows:,}")

assert verified_shards == EXPECTED_SHARDS
assert verified_rows == EXPECTED_S2_ROWS

# ------------------------------------------------------------------------------
# 5. VERIFY S1 EMBEDDING CHECKPOINT STILL EXISTS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. VERIFY PREVIOUS S1 EMBEDDINGS")
print("-" * 78)

s1_files = [
    p for p in S1_EMBED_ROOT.rglob("*")
    if p.is_file()
]

print(f"S1 embedding files: {len(s1_files):,}")

assert len(s1_files) > 0

# ------------------------------------------------------------------------------
# 6. COPY MANIFEST + CREATE STATE MANIFEST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. WRITE CHECKPOINT METADATA")
print("-" * 78)

import shutil

shutil.copy2(
    manifest_path,
    MANIFEST_COPY,
)

metadata = {
    "checkpoint_name":
        "AMLC2026_AFTER_CELL65_S2_EMBEDDINGS",

    "state":
        "CELL65_COMPLETE_TRAIN_S2_EMBEDDINGS_COMPLETE",

    "next_step":
        "TRAIN S3 EMBEDDINGS",

    "feature_root":
        str(FEATURE_ROOT),

    "s2_record_path":
        str(S2_RECORD_PATH),

    "s1_embedding_root":
        str(S1_EMBED_ROOT),

    "s2_embedding_root":
        str(S2_EMBED_ROOT),

    "model":
        MODEL_NAME,

    "embedding_dimension":
        EMBED_DIM,

    "dtype":
        "float16",

    "normalized":
        True,

    "s2_rows":
        EXPECTED_S2_ROWS,

    "s2_shards":
        EXPECTED_SHARDS,

    "s2_shard_rows":
        SHARD_ROWS,

    "verified_s2_rows":
        verified_rows,

    "s1_rows":
        EXPECTED_S1_ROWS,

    "s2_embedding_bytes":
        total_embedding_bytes,

    "s2_id_bytes":
        total_id_bytes,

    "created_at":
        time.strftime("%Y-%m-%d %H:%M:%S"),

    "previous_state":
        "CELL64_COMPLETE_TRAIN_S1_EMBEDDINGS_COMPLETE",
}

with open(METADATA_PATH, "w") as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

# ------------------------------------------------------------------------------
# 7. ENVIRONMENT SNAPSHOT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. SAVE ENVIRONMENT")
print("-" * 78)

try:

    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        timeout=120,
    )

    ENV_PATH.write_text(result.stdout)

except Exception as e:

    ENV_PATH.write_text(
        f"pip freeze failed: {repr(e)}\n"
    )

# ------------------------------------------------------------------------------
# 8. COMPLETION MARKER
# ------------------------------------------------------------------------------

MARKER_PATH.write_text(
    "CELL 65 COMPLETE\n"
    "TRAIN S2 EMBEDDINGS COMPLETE\n"
    f"ROWS={EXPECTED_S2_ROWS}\n"
    f"SHARDS={EXPECTED_SHARDS}\n"
    f"EMBED_DIM={EMBED_DIM}\n"
    f"MODEL={MODEL_NAME}\n"
)

# ------------------------------------------------------------------------------
# 9. FINAL
# ------------------------------------------------------------------------------

elapsed = (
    time.time() - T0
) / 60

print("\n" + "=" * 78)
print("✅ CELL 65 CHECKPOINT COMPLETE")
print("=" * 78)

print(f"""
CHECKPOINT:
  {CHECKPOINT_ROOT}

S2 EMBEDDINGS:
  {S2_EMBED_ROOT}

Rows:
  {verified_rows:,}

Shards:
  {verified_shards}/{EXPECTED_SHARDS}

Embedding storage:
  {total_embedding_bytes / (1024**3):.2f} GB

ID storage:
  {total_id_bytes / (1024**3):.2f} GB

Model:
  {MODEL_NAME}

Dimension:
  {EMBED_DIM}

✅ S1 embeddings still present
✅ S2 embeddings fully verified
✅ Manifest copied
✅ Metadata written
✅ Environment snapshot written
✅ Completion marker written

NEXT:
  CELL 66 — TRAIN S3 EMBEDDINGS

Checkpoint runtime:
  {elapsed:.2f} min
""")

AMLC 2026 — CHECKPOINT AFTER CELL 65

------------------------------------------------------------------------------
1. FEATURE LAKE
------------------------------------------------------------------------------
Feature root: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1
S2 embeddings: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s2/full_record

------------------------------------------------------------------------------
2. VERIFY S2 RECORD TABLE
------------------------------------------------------------------------------
S2 record rows: 5,034,616

------------------------------------------------------------------------------
3. VERIFY S2 EMBEDDING MANIFEST
------------------------------------------------------------------------------
{
  "status": "COMPLETE",
  "source": "train/s2",
  "rows": 5034616,
  "embedding_model": "intfloat/multilingual-e5-small",
  "embedding_dimension": 384,
  "max_sequence_length": 256,
  "text_recipe": "passage_v1_name_address_count

In [15]:
from pathlib import Path

TARGET = "FINAL_FEATURE_LAKE_V1"

candidates = []

# Current writable session
p = Path("/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1")
if p.exists():
    candidates.append(p)

# Attached notebook outputs / datasets
for base in [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]:
    if base.exists():
        for p in base.rglob(TARGET):
            if p.is_dir():
                candidates.append(p)

# Deduplicate
candidates = list(dict.fromkeys(candidates))

print("Feature lake candidates:")
for p in candidates:
    print("  ", p)

assert candidates, (
    "FINAL_FEATURE_LAKE_V1 was not found. "
    "Attach the saved Notebook Output via "
    "Add Data → Notebook Output Files."
)

# Prefer writable working copy if present, otherwise attached output.
working = [
    p for p in candidates
    if str(p).startswith("/kaggle/working/")
]

FEATURE_ROOT = (
    working[0]
    if working
    else candidates[0]
)

print("\n✅ FEATURE_ROOT:")
print(FEATURE_ROOT)

Feature lake candidates:
   /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

✅ FEATURE_ROOT:
/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1


In [16]:
# ==============================================================================
# AMLC 2026 — CELL 66
# TRAIN / S3 GPU SEMANTIC EMBEDDINGS
#
# FIRST:
#   Aggressive RAM cleanup from previous cells.
#
# THEN:
#   Train/S3 -> multilingual-e5-small -> 384d float16
#
# INPUT:
#   FINAL_FEATURE_LAKE_V1/records/train/s3/records.parquet
#
# OUTPUT:
#   FINAL_FEATURE_LAKE_V1/embeddings/train/s3/full_record/
#
# DESIGN:
#   - 100k-row durable shards
#   - float16 embeddings
#   - aligned entity_id parquet
#   - resumable after kernel reset
#   - existing completed shards are skipped
# ==============================================================================

from pathlib import Path
import gc
import json
import math
import os
import time

# ------------------------------------------------------------------------------
# 0. RAM CLEANUP — DO THIS BEFORE LOADING ANYTHING LARGE
# ------------------------------------------------------------------------------

print("=" * 78)
print("AMLC 2026 — CELL 66")
print("TRAIN / S3 GPU SEMANTIC EMBEDDINGS")
print("=" * 78)

print("\n" + "-" * 78)
print("0. PRE-RUN RAM CLEANUP")
print("-" * 78)

# Objects created by previous cells that are no longer needed for embedding.
# The durable versions of the important artifacts are already on disk.
BIG_RUNTIME_OBJECTS = [
    # Pair / GT state
    "pair_sample",
    "sampled",
    "cand",
    "labeled",
    "gt",
    "gt_raw",
    "gt_edges",
    "gt_edges_full",
    "positive_edges",
    "sample_positive_edges",
    "negative_pairs",
    "positive_pairs",
    "val_s1",
    "val_s1_df",

    # Record dataframes
    "s1_df",
    "s2_df",
    "s3_df",
    "train_s1",
    "train_s2",
    "train_s3",
    "test_s1",
    "test_s2",
    "test_s3",
    "records",

    # Feature / embedding arrays
    "E",
    "embedding",
    "embeddings",
    "feats",
    "features",
    "X",
    "X_train",
    "X_val",
    "X_test",
    "texts",
    "entity_ids",

    # Old model / tokenizer references.
    # Cell 66 reloads the model cleanly after memory cleanup.
    "embed_model",
    "model",
    "tokenizer",

    # Common temporary objects
    "shard_df",
    "smoke",
    "smoke_texts",
    "df",
    "arr",
]

deleted = []

for name in BIG_RUNTIME_OBJECTS:
    if name in globals():
        try:
            del globals()[name]
            deleted.append(name)
        except Exception:
            pass

print(
    f"Deleted large runtime objects: {len(deleted):,}"
)

if deleted:
    print("Examples:")
    print("  " + ", ".join(deleted[:20]))

# Python garbage collector.
gc.collect()

# CUDA cache.
try:
    import torch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

except Exception as e:
    print(f"CUDA cleanup note: {e}")

# Optional RAM report.
try:
    import psutil

    vm = psutil.virtual_memory()

    print(
        f"\nRAM after cleanup:"
        f" {vm.used / (1024**3):.2f} / "
        f"{vm.total / (1024**3):.2f} GB "
        f"({vm.percent:.1f}%)"
    )

except Exception:
    pass

# ------------------------------------------------------------------------------
# 1. PATH DISCOVERY
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. FEATURE-LAKE DISCOVERY")
print("-" * 78)

FEATURE_TARGET = "FINAL_FEATURE_LAKE_V1"

feature_candidates = []

# Writable working copy first.
working_feature_root = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

if working_feature_root.exists():
    feature_candidates.append(
        working_feature_root
    )

# Attached notebook output / dataset copy.
kaggle_input = Path("/kaggle/input")

if kaggle_input.exists():
    for p in kaggle_input.rglob(
        FEATURE_TARGET
    ):
        if p.is_dir():
            feature_candidates.append(p)

# Deduplicate while preserving order.
feature_candidates = list(
    dict.fromkeys(feature_candidates)
)

print("Feature lake candidates:")

for p in feature_candidates:
    print(f"  - {p}")

assert feature_candidates, (
    "FINAL_FEATURE_LAKE_V1 was not found.\n\n"
    "Attach the saved notebook output containing the feature lake "
    "through Kaggle -> Add Data -> Notebook Output Files."
)

working_candidates = [
    p for p in feature_candidates
    if str(p).startswith("/kaggle/working/")
]

FEATURE_ROOT = (
    working_candidates[0]
    if working_candidates
    else feature_candidates[0]
)

print(f"\n✅ FEATURE_ROOT = {FEATURE_ROOT}")

# ------------------------------------------------------------------------------
# 2. PATHS / CONSTANTS
# ------------------------------------------------------------------------------

S3_RECORD_PATH = (
    FEATURE_ROOT
    / "records"
    / "train"
    / "s3"
    / "records.parquet"
)

S3_EMBED_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s3"
    / "full_record"
)

S3_EMBED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

S3_MANIFEST_PATH = (
    S3_EMBED_ROOT
    / "manifest.json"
)

S3_COMPLETE_MARKER = (
    S3_EMBED_ROOT
    / "COMPLETE"
)

MODEL_NAME = (
    "intfloat/multilingual-e5-small"
)

EXPECTED_ROWS = 5_285_603

EMBED_DIM = 384
MAX_LENGTH = 256

SHARD_ROWS = 100_000
ENCODE_BATCH_SIZE = 256

TEXT_RECIPE = (
    "passage_v1_name_address_country"
)

print(f"\nS3 records   : {S3_RECORD_PATH}")
print(f"S3 embeddings: {S3_EMBED_ROOT}")

assert S3_RECORD_PATH.exists(), (
    f"S3 record table missing:\n{S3_RECORD_PATH}"
)

# ------------------------------------------------------------------------------
# 3. GPU CHECK
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. GPU")
print("-" * 78)

import torch

assert torch.cuda.is_available(), (
    "CUDA is unavailable."
)

DEVICE = "cuda"

GPU_NAME = torch.cuda.get_device_name(0)

GPU_VRAM_GB = (
    torch.cuda.get_device_properties(0)
    .total_memory
    / (1024**3)
)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", GPU_NAME)
print(f"VRAM GB: {GPU_VRAM_GB:.2f}")

# ------------------------------------------------------------------------------
# 4. S3 RECORD PREFLIGHT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. TRAIN / S3 RECORD TABLE")
print("-" * 78)

schema = pl.read_parquet_schema(
    S3_RECORD_PATH
)

print("Columns:")

for c in schema:
    print(f"  - {c}")

REQUIRED_COLUMNS = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
]

missing = [
    c for c in REQUIRED_COLUMNS
    if c not in schema
]

assert not missing, (
    f"Missing required columns: {missing}"
)

record_rows = (
    pl.scan_parquet(S3_RECORD_PATH)
    .select(pl.len())
    .collect()
    .item()
)

print(
    f"\nRows: {record_rows:,}"
)

assert record_rows == EXPECTED_ROWS, (
    f"S3 row mismatch: "
    f"{record_rows:,} != {EXPECTED_ROWS:,}"
)

# ------------------------------------------------------------------------------
# 5. LOAD MODEL
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. LOAD MULTILINGUAL EMBEDDING MODEL")
print("-" * 78)

from sentence_transformers import SentenceTransformer

print(
    f"Loading {MODEL_NAME} onto CUDA..."
)

embed_model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
)

try:
    embed_model.max_seq_length = MAX_LENGTH
except Exception:
    pass

try:
    detected_dim = (
        embed_model.get_embedding_dimension()
    )
except AttributeError:
    detected_dim = (
        embed_model.get_sentence_embedding_dimension()
    )

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print(
    "Embedding dimension:",
    detected_dim,
)

assert detected_dim == EMBED_DIM

# ------------------------------------------------------------------------------
# 6. CANONICAL TEXT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. CANONICAL RECORD TEXT")
print("-" * 78)

print(
    "Recipe:",
    TEXT_RECIPE,
)

print(
    "Format:",
    "passage: name=<business_name> | "
    "address=<business_address> | "
    "country=<country>",
)

def build_record_text(df: pl.DataFrame) -> list[str]:

    return (
        df
        .with_columns([
            pl.col("business_name")
            .fill_null("")
            .cast(pl.Utf8)
            .alias("_name"),

            pl.col("business_address")
            .fill_null("")
            .cast(pl.Utf8)
            .alias("_address"),

            pl.col("country")
            .fill_null("")
            .cast(pl.Utf8)
            .alias("_country"),
        ])
        .with_columns(
            pl.concat_str(
                [
                    pl.lit("passage: name="),
                    pl.col("_name"),
                    pl.lit(" | address="),
                    pl.col("_address"),
                    pl.lit(" | country="),
                    pl.col("_country"),
                ],
                separator="",
            ).alias("_text")
        )
        .select("_text")
        .get_column("_text")
        .to_list()
    )

# ------------------------------------------------------------------------------
# 7. TEXT PREFLIGHT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. TEXT PREFLIGHT")
print("-" * 78)

smoke = (
    pl.scan_parquet(S3_RECORD_PATH)
    .select(REQUIRED_COLUMNS)
    .head(3)
    .collect()
)

smoke_texts = build_record_text(smoke)

for i, txt in enumerate(smoke_texts):
    print(f"\n[{i}]")
    print(txt[:1000])

assert len(smoke_texts) == 3
assert all(
    isinstance(x, str)
    for x in smoke_texts
)

del smoke
del smoke_texts

gc.collect()

print("\n✅ Text construction passed.")

# ------------------------------------------------------------------------------
# 8. SHARD PLAN
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. SHARD PLAN")
print("-" * 78)

n_shards = math.ceil(
    EXPECTED_ROWS / SHARD_ROWS
)

print(
    f"Rows per shard : {SHARD_ROWS:,}"
)

print(
    f"Total rows     : {EXPECTED_ROWS:,}"
)

print(
    f"Total shards   : {n_shards:,}"
)

final_shard_rows = (
    EXPECTED_ROWS
    - (n_shards - 1) * SHARD_ROWS
)

print(
    f"Final shard rows: {final_shard_rows:,}"
)

# Expected:
# 53 shards
# final shard = 85,603 rows

# ------------------------------------------------------------------------------
# 9. VERIFY EXISTING SHARDS FOR RESUME
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("8. RESUME SCAN")
print("-" * 78)

def verify_shard(
    shard_index: int,
    expected_rows: int,
) -> bool:

    emb_path = (
        S3_EMBED_ROOT
        / f"embeddings_{shard_index:04d}.npy"
    )

    id_path = (
        S3_EMBED_ROOT
        / f"ids_{shard_index:04d}.parquet"
    )

    if (
        not emb_path.exists()
        or not id_path.exists()
    ):
        return False

    try:

        arr = np.load(
            emb_path,
            mmap_mode="r",
        )

        ids = pl.read_parquet(
            id_path,
            columns=["entity_id"],
        )

        ok = (
            arr.shape
            == (
                expected_rows,
                EMBED_DIM,
            )
            and ids.height == expected_rows
            and arr.dtype == np.float16
        )

        del arr
        del ids

        return bool(ok)

    except Exception:
        return False


completed = set()

for shard_idx in range(n_shards):

    start = shard_idx * SHARD_ROWS

    end = min(
        start + SHARD_ROWS,
        EXPECTED_ROWS,
    )

    expected_rows = end - start

    if verify_shard(
        shard_idx,
        expected_rows,
    ):
        completed.add(shard_idx)

print(
    f"Valid existing shards: "
    f"{len(completed):,} / {n_shards:,}"
)

if completed:
    print(
        "Existing completed shard IDs:",
        sorted(completed)[:20],
        "..."
        if len(completed) > 20
        else "",
    )

# ------------------------------------------------------------------------------
# 10. EMBED S3 SHARDS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("9. EMBEDDING TRAIN / S3")
print("-" * 78)

for shard_idx in range(n_shards):

    shard_start = (
        shard_idx
        * SHARD_ROWS
    )

    shard_end = min(
        shard_start
        + SHARD_ROWS,
        EXPECTED_ROWS,
    )

    shard_rows = (
        shard_end
        - shard_start
    )

    emb_path = (
        S3_EMBED_ROOT
        / f"embeddings_{shard_idx:04d}.npy"
    )

    id_path = (
        S3_EMBED_ROOT
        / f"ids_{shard_idx:04d}.parquet"
    )

    # --------------------------------------------------------------------------
    # Already completed.
    # --------------------------------------------------------------------------

    if shard_idx in completed:

        print(
            f"[SKIP] "
            f"{shard_idx + 1}/{n_shards} "
            f"rows={shard_start:,}:{shard_end:,}"
        )

        continue

    # Remove stale partial outputs.
    if emb_path.exists():
        emb_path.unlink()

    if id_path.exists():
        id_path.unlink()

    print(
        f"\n[S3 {shard_idx + 1}/{n_shards}] "
        f"rows={shard_start:,}:{shard_end:,}"
    )

    shard_t0 = time.time()

    # --------------------------------------------------------------------------
    # Read exactly this shard.
    # --------------------------------------------------------------------------

    shard_df = (
        pl.scan_parquet(S3_RECORD_PATH)
        .select(REQUIRED_COLUMNS)
        .slice(
            shard_start,
            shard_rows,
        )
        .collect()
    )

    assert shard_df.height == shard_rows

    # --------------------------------------------------------------------------
    # Preserve exact ID ordering.
    # --------------------------------------------------------------------------

    shard_entity_ids = (
        shard_df
        .get_column("entity_id")
        .cast(pl.Utf8)
        .to_list()
    )

    assert (
        len(shard_entity_ids)
        == shard_rows
    )

    # --------------------------------------------------------------------------
    # Build text.
    # --------------------------------------------------------------------------

    texts = build_record_text(
        shard_df
    )

    assert len(texts) == shard_rows

    # --------------------------------------------------------------------------
    # GPU ENCODE
    # --------------------------------------------------------------------------

    with torch.inference_mode():

        E = embed_model.encode(
            texts,
            batch_size=ENCODE_BATCH_SIZE,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
            convert_to_tensor=False,
            device=DEVICE,
        )

    # Float16 durable storage.
    E = np.asarray(
        E,
        dtype=np.float16,
    )

    # --------------------------------------------------------------------------
    # HARD VALIDATION
    # --------------------------------------------------------------------------

    assert E.shape == (
        shard_rows,
        EMBED_DIM,
    ), (
        f"Bad embedding shape "
        f"{E.shape} for shard {shard_idx}"
    )

    assert np.isfinite(E).all(), (
        f"Non-finite embedding found "
        f"in shard {shard_idx}"
    )

    # --------------------------------------------------------------------------
    # SAVE IDS
    # --------------------------------------------------------------------------

    ids_df = pl.DataFrame({
        "entity_id":
            shard_entity_ids
    })

    ids_df.write_parquet(
        id_path,
        compression="zstd",
    )

    # --------------------------------------------------------------------------
    # SAVE EMBEDDINGS
    # --------------------------------------------------------------------------

    np.save(
        emb_path,
        E,
        allow_pickle=False,
    )

    # --------------------------------------------------------------------------
    # RELEASE RAM / VRAM BEFORE VALIDATION
    # --------------------------------------------------------------------------

    del shard_df
    del shard_entity_ids
    del texts
    del ids_df
    del E

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------------------------
    # Verify saved shard.
    # --------------------------------------------------------------------------

    assert verify_shard(
        shard_idx,
        shard_rows,
    ), (
        f"Saved shard {shard_idx} "
        f"failed verification."
    )

    completed.add(
        shard_idx
    )

    elapsed = (
        time.time()
        - shard_t0
    )

    rate = (
        shard_rows
        / max(elapsed, 1e-9)
    )

    print(
        f"[DONE] "
        f"{shard_start:,}:{shard_end:,} "
        f"rows={shard_rows:,} "
        f"rate={rate:,.0f}/sec "
        f"time={elapsed / 60:.2f}m"
    )

    # --------------------------------------------------------------------------
    # Durable progress manifest after EVERY shard.
    # --------------------------------------------------------------------------

    completed_rows = sum(
        min(
            SHARD_ROWS,
            EXPECTED_ROWS
            - i * SHARD_ROWS,
        )
        for i in completed
    )

    progress = {

        "status": "RUNNING",

        "source":
            "train/s3",

        "model":
            MODEL_NAME,

        "embedding_dimension":
            EMBED_DIM,

        "max_sequence_length":
            MAX_LENGTH,

        "text_recipe":
            TEXT_RECIPE,

        "expected_rows":
            EXPECTED_ROWS,

        "shard_rows":
            SHARD_ROWS,

        "total_shards":
            n_shards,

        "completed_shards":
            sorted(
                int(x)
                for x in completed
            ),

        "completed_rows":
            int(completed_rows),

        "last_completed_shard":
            int(shard_idx),

        "updated_at":
            time.strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
    }

    with open(
        S3_MANIFEST_PATH,
        "w",
    ) as f:

        json.dump(
            progress,
            f,
            indent=2,
        )

# ------------------------------------------------------------------------------
# 11. FINAL VERIFICATION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("10. FINAL S3 VERIFICATION")
print("-" * 78)

assert len(completed) == n_shards, (
    f"Only {len(completed)} / "
    f"{n_shards} shards complete."
)

verified_rows = 0

for shard_idx in range(n_shards):

    start = (
        shard_idx
        * SHARD_ROWS
    )

    end = min(
        start
        + SHARD_ROWS,
        EXPECTED_ROWS,
    )

    shard_rows = end - start

    assert verify_shard(
        shard_idx,
        shard_rows,
    )

    verified_rows += shard_rows

print(
    f"Verified shards: "
    f"{len(completed):,}"
)

print(
    f"Verified embedding rows: "
    f"{verified_rows:,}"
)

assert verified_rows == EXPECTED_ROWS

# ------------------------------------------------------------------------------
# 12. FINAL MANIFEST
# ------------------------------------------------------------------------------

final_manifest = {

    "status":
        "COMPLETE",

    "source":
        "train/s3",

    "rows":
        EXPECTED_ROWS,

    "embedding_model":
        MODEL_NAME,

    "embedding_dimension":
        EMBED_DIM,

    "max_sequence_length":
        MAX_LENGTH,

    "text_recipe":
        TEXT_RECIPE,

    "shard_rows":
        SHARD_ROWS,

    "shards":
        n_shards,

    "dtype":
        "float16",

    "normalized":
        True,

    "gpu":
        GPU_NAME,

    "torch":
        str(torch.__version__),

    "cuda":
        str(torch.version.cuda),

    "output_root":
        str(S3_EMBED_ROOT),

    "completed_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

with open(
    S3_MANIFEST_PATH,
    "w",
) as f:

    json.dump(
        final_manifest,
        f,
        indent=2,
    )

# ------------------------------------------------------------------------------
# 13. COMPLETION MARKER
# ------------------------------------------------------------------------------

S3_COMPLETE_MARKER.write_text(
    "CELL 66 COMPLETE\n"
    "TRAIN S3 EMBEDDINGS COMPLETE\n"
    f"ROWS={EXPECTED_ROWS}\n"
    f"SHARDS={n_shards}\n"
    f"EMBED_DIM={EMBED_DIM}\n"
    f"MODEL={MODEL_NAME}\n"
)

# ------------------------------------------------------------------------------
# 14. FINAL STORAGE REPORT
# ------------------------------------------------------------------------------

embedding_bytes = sum(
    p.stat().st_size
    for p in S3_EMBED_ROOT.glob(
        "embeddings_*.npy"
    )
)

id_bytes = sum(
    p.stat().st_size
    for p in S3_EMBED_ROOT.glob(
        "ids_*.parquet"
    )
)

total_minutes = (
    time.time() - T0
) / 60

# RAM report.
try:
    import psutil

    vm = psutil.virtual_memory()

    ram_line = (
        f"{vm.used / (1024**3):.2f} / "
        f"{vm.total / (1024**3):.2f} GB "
        f"({vm.percent:.1f}%)"
    )

except Exception:
    ram_line = "unavailable"

print("\n" + "=" * 78)
print("✅ CELL 66 COMPLETE — TRAIN S3 GPU EMBEDDINGS")
print("=" * 78)

print(f"""
Rows:
  {EXPECTED_ROWS:,}

Embedding dimension:
  {EMBED_DIM}

Model:
  {MODEL_NAME}

GPU:
  {GPU_NAME}

Shards:
  {n_shards:,}

Shard size:
  {SHARD_ROWS:,}

Final shard:
  {final_shard_rows:,}

Embedding dtype:
  float16

Embedding storage:
  {embedding_bytes / (1024**3):.2f} GB

ID storage:
  {id_bytes / (1024**3):.2f} GB

Output:
  {S3_EMBED_ROOT}

Manifest:
  {S3_MANIFEST_PATH}

Completion marker:
  {S3_COMPLETE_MARKER}

RAM at finish:
  {ram_line}

✅ Every S3 shard verified.
✅ All 5,285,603 S3 records embedded.
✅ Embeddings normalized.
✅ Durable progress manifest written.
✅ Kernel-reset safe.

NEXT:
  CHECKPOINT AFTER CELL 66
""")

print(
    f"Cell 66 runtime: "
    f"{total_minutes:.2f} min"
)

AMLC 2026 — CELL 66
TRAIN / S3 GPU SEMANTIC EMBEDDINGS

------------------------------------------------------------------------------
0. PRE-RUN RAM CLEANUP
------------------------------------------------------------------------------
Deleted large runtime objects: 13
Examples:
  sampled, cand, labeled, gt, gt_raw, gt_edges, gt_edges_full, positive_edges, negative_pairs, positive_pairs, val_s1, val_s1_df, embed_model

RAM after cleanup: 26.21 / 31.35 GB (78.2%)

------------------------------------------------------------------------------
1. FEATURE-LAKE DISCOVERY
------------------------------------------------------------------------------
Feature lake candidates:
  - /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

✅ FEATURE_ROOT = /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

S3 records   : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s3/records.parquet
S3 embeddings: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s3/full_record

---------------

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: intfloat/multilingual-e5-small
Device: cuda
Embedding dimension: 384

------------------------------------------------------------------------------
5. CANONICAL RECORD TEXT
------------------------------------------------------------------------------
Recipe: passage_v1_name_address_country
Format: passage: name=<business_name> | address=<business_address> | country=<country>

------------------------------------------------------------------------------
6. TEXT PREFLIGHT
------------------------------------------------------------------------------

[0]
passage: name=wilfordhancock.com | address=Mack Rd, Haltom City, Texas | country=US

[1]
passage: name=International South Consultants Private Ltd | address= | country=India

[2]
passage: name=LLC Moncada Léarning Center | address=5780 Fawn Ct, Fort Worth, Texas | country=US

✅ Text construction passed.

------------------------------------------------------------------------------
7. SHARD PLAN
--------------------------------

Batches:   0%|          | 0/391 [00:00<?, ?it/s]

...

[DONE] 5,000,000:5,100,000 rows=100,000 rate=1,210/sec time=1.38m

[S3 52/53] rows=5,100,000:5,200,000


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

[DONE] 5,100,000:5,200,000 rows=100,000 rate=1,208/sec time=1.38m

[S3 53/53] rows=5,200,000:5,285,603


Batches:   0%|          | 0/335 [00:00<?, ?it/s]

[DONE] 5,200,000:5,285,603 rows=85,603 rate=1,214/sec time=1.18m

------------------------------------------------------------------------------
10. FINAL S3 VERIFICATION
------------------------------------------------------------------------------
Verified shards: 53
Verified embedding rows: 5,285,603

✅ CELL 66 COMPLETE — TRAIN S3 GPU EMBEDDINGS

Rows:
  5,285,603

Embedding dimension:
  384

Model:
  intfloat/multilingual-e5-small

GPU:
  Tesla T4

Shards:
  53

Shard size:
  100,000

Final shard:
  85,603

Embedding dtype:
  float16

Embedding storage:
  3.78 GB

ID storage:
  0.03 GB

Output:
  /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s3/full_record

Manifest:
  /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s3/full_record/manifest.json

Completion marker:
  /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s3/full_record/COMPLETE

RAM at finish:
  24.56 / 31.35 GB (78.9%)

✅ Every S3 shard verified.
✅ All 5,285,603 S3 records

In [17]:
# ==============================================================================
# AMLC 2026 — CELL 66A
# DIAGNOSE ACTUAL CELL-64 S1 EMBEDDING ARTIFACT LAYOUT
# ==============================================================================

from pathlib import Path
import json
import os

import numpy as np
import polars as pl

print("=" * 78)
print("AMLC 2026 — CELL 66A")
print("DIAGNOSE ACTUAL TRAIN S1 EMBEDDING ARTIFACT")
print("=" * 78)

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

S1_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s1"
    / "full_record"
)

assert S1_ROOT.exists(), (
    f"S1 embedding directory missing:\n{S1_ROOT}"
)

# ------------------------------------------------------------------------------
# 1. COMPLETE FILE INVENTORY
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. FILE INVENTORY")
print("-" * 78)

files = sorted(
    [p for p in S1_ROOT.rglob("*") if p.is_file()],
    key=lambda p: str(p),
)

print(f"Total files: {len(files):,}")

for i, p in enumerate(files):

    size_mb = p.stat().st_size / (1024 ** 2)

    print(
        f"{i:03d} | "
        f"{p.name:<45} | "
        f"{p.suffix:<12} | "
        f"{size_mb:10.2f} MB"
    )

# ------------------------------------------------------------------------------
# 2. FILE EXTENSION SUMMARY
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. EXTENSION SUMMARY")
print("-" * 78)

ext_counts = {}

for p in files:
    ext = p.suffix.lower() or "<none>"
    ext_counts[ext] = (
        ext_counts.get(ext, 0) + 1
    )

for ext, count in sorted(ext_counts.items()):
    print(
        f"{ext:<15} : {count:>5}"
    )

# ------------------------------------------------------------------------------
# 3. MANIFEST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. MANIFEST")
print("-" * 78)

manifest_path = S1_ROOT / "manifest.json"

if manifest_path.exists():

    print(
        f"Manifest: {manifest_path}"
    )

    with open(manifest_path) as f:
        manifest = json.load(f)

    print(
        json.dumps(
            manifest,
            indent=2,
        )
    )

else:

    print("NO manifest.json")

# ------------------------------------------------------------------------------
# 4. INSPECT ALL NUMPY ARRAYS WITHOUT LOADING THEM
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. NUMPY ARRAY INSPECTION")
print("-" * 78)

npy_files = [
    p for p in files
    if p.suffix.lower() == ".npy"
]

print(
    f"NPY files: {len(npy_files):,}"
)

for p in npy_files:

    try:

        arr = np.load(
            p,
            mmap_mode="r",
        )

        print(
            f"{p.name:<45} "
            f"shape={str(arr.shape):<24} "
            f"dtype={arr.dtype}"
        )

        del arr

    except Exception as e:

        print(
            f"{p.name:<45} "
            f"ERROR={repr(e)}"
        )

# ------------------------------------------------------------------------------
# 5. INSPECT ALL NPZ ARRAYS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. NPZ INSPECTION")
print("-" * 78)

npz_files = [
    p for p in files
    if p.suffix.lower() == ".npz"
]

print(
    f"NPZ files: {len(npz_files):,}"
)

for p in npz_files:

    try:

        z = np.load(
            p,
            mmap_mode="r",
        )

        print(
            f"\n{p.name}"
        )

        for key in z.files:

            arr = z[key]

            print(
                f"  {key}: "
                f"shape={arr.shape} "
                f"dtype={arr.dtype}"
            )

        del z

    except Exception as e:

        print(
            f"{p.name}: ERROR={repr(e)}"
        )

# ------------------------------------------------------------------------------
# 6. INSPECT ALL PARQUET FILES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. PARQUET INSPECTION")
print("-" * 78)

parquet_files = [
    p for p in files
    if p.suffix.lower() == ".parquet"
]

print(
    f"Parquet files: {len(parquet_files):,}"
)

for p in parquet_files:

    try:

        schema = pl.read_parquet_schema(p)

        rows = (
            pl.scan_parquet(p)
            .select(pl.len())
            .collect()
            .item()
        )

        print(
            f"\n{p.name}"
        )

        print(
            f"  rows: {rows:,}"
        )

        print(
            "  columns:",
            schema,
        )

        # Inspect only one row.
        sample = (
            pl.read_parquet(
                p,
                n_rows=1,
            )
        )

        print(
            "  sample columns:",
            sample.columns,
        )

        print(
            "  sample schema:",
            sample.schema,
        )

        del sample

    except Exception as e:

        print(
            f"{p.name}: ERROR={repr(e)}"
        )

# ------------------------------------------------------------------------------
# 7. OTHER BINARY FILE TYPES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. OTHER FILE TYPES")
print("-" * 78)

known = {
    ".json",
    ".npy",
    ".npz",
    ".parquet",
    ".txt",
    ".csv",
}

for p in files:

    if p.suffix.lower() not in known:

        size_mb = (
            p.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"{p.name:<50} "
            f"{p.suffix:<12} "
            f"{size_mb:.2f} MB"
        )

# ------------------------------------------------------------------------------
# 8. POTENTIAL EMBEDDING FILE HEURISTICS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("8. EMBEDDING-LAYOUT HEURISTICS")
print("-" * 78)

for p in files:

    name = p.name.lower()

    if any(
        token in name
        for token in [
            "embed",
            "vector",
            "feat",
            "chunk",
            "shard",
            "full",
        ]
    ):

        print(
            f"Potential embedding artifact: "
            f"{p.relative_to(S1_ROOT)}"
        )

print("\n" + "=" * 78)
print("✅ DIAGNOSTIC COMPLETE")
print("=" * 78)

print("""
NO FILES WERE MODIFIED.
NO EMBEDDINGS WERE RECOMPUTED.
NO LARGE EMBEDDING ARRAYS WERE LOADED INTO RAM.

Paste the output of this cell.
Then I will give you the exact S1 verification/checkpoint cell
for the format Cell 64 actually produced.
""")

AMLC 2026 — CELL 66A
DIAGNOSE ACTUAL TRAIN S1 EMBEDDING ARTIFACT

------------------------------------------------------------------------------
1. FILE INVENTORY
------------------------------------------------------------------------------
Total files: 47
000 | emb_000000000_000100000.npy                   | .npy         |      73.24 MB
001 | emb_000100000_000200000.npy                   | .npy         |      73.24 MB
002 | emb_000200000_000300000.npy                   | .npy         |      73.24 MB
003 | emb_000300000_000400000.npy                   | .npy         |      73.24 MB
004 | emb_000400000_000500000.npy                   | .npy         |      73.24 MB
005 | emb_000500000_000600000.npy                   | .npy         |      73.24 MB
006 | emb_000600000_000700000.npy                   | .npy         |      73.24 MB
007 | emb_000700000_000800000.npy                   | .npy         |      73.24 MB
008 | emb_000800000_000900000.npy                   | .npy         |      73.2

In [18]:
# ==============================================================================
# AMLC 2026 — MASTER CHECKPOINT AFTER CELL 66
# FINAL FIX — SUPPORTS ORIGINAL CELL-64 S1 FORMAT
#
# IMPORTANT:
#   - DOES NOT recompute embeddings
#   - DOES NOT rewrite embedding shards
#   - DOES NOT create ZIP
#   - ONLY verifies existing artifacts + creates durable markers/metadata
# ==============================================================================

from pathlib import Path
import gc
import json
import subprocess
import sys
import time

import numpy as np
import polars as pl

print("=" * 78)
print("AMLC 2026 — MASTER CHECKPOINT AFTER CELL 66")
print("FINAL FIX — ORIGINAL CELL-64 S1 FORMAT")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# 0. RAM CLEANUP
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("0. RAM CLEANUP")
print("-" * 78)

for name in [
    "embed_model",
    "model",
    "tokenizer",
    "E",
    "embedding",
    "embeddings",
    "texts",
    "shard_df",
    "smoke",
    "smoke_texts",
    "df",
    "arr",
]:
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass

gc.collect()

try:
    import torch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

try:
    import psutil

    vm = psutil.virtual_memory()

    print(
        f"RAM: {vm.used / (1024**3):.2f} / "
        f"{vm.total / (1024**3):.2f} GB "
        f"({vm.percent:.1f}%)"
    )
except Exception:
    pass

# ------------------------------------------------------------------------------
# 1. LOCATE FEATURE LAKE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. LOCATE FEATURE LAKE")
print("-" * 78)

FEATURE_NAME = "FINAL_FEATURE_LAKE_V1"

candidates = []

working_root = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

if working_root.exists():
    candidates.append(
        working_root
    )

input_root = Path(
    "/kaggle/input"
)

if input_root.exists():

    for p in input_root.rglob(
        FEATURE_NAME
    ):

        if p.is_dir():
            candidates.append(p)

candidates = list(
    dict.fromkeys(candidates)
)

print("Candidates:")

for p in candidates:
    print("  -", p)

assert candidates, (
    "FINAL_FEATURE_LAKE_V1 not found. "
    "Attach the saved Notebook Output."
)

working_candidates = [
    p for p in candidates
    if str(p).startswith(
        "/kaggle/working/"
    )
]

FEATURE_ROOT = (
    working_candidates[0]
    if working_candidates
    else candidates[0]
)

print(
    "\n✅ FEATURE_ROOT =",
    FEATURE_ROOT
)

# ------------------------------------------------------------------------------
# 2. VERIFY RECORD TABLES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. VERIFY RECORD LAKE")
print("-" * 78)

EXPECTED_RECORDS = {
    ("train", "s1"): 2_206_821,
    ("train", "s2"): 5_034_616,
    ("train", "s3"): 5_285_603,
    ("test", "s1"): 1_732_544,
    ("test", "s2"): 4_887_273,
    ("test", "s3"): 5_082_316,
}

record_inventory = {}

for (split, source), expected_rows in EXPECTED_RECORDS.items():

    path = (
        FEATURE_ROOT
        / "records"
        / split
        / source
        / "records.parquet"
    )

    assert path.exists(), (
        f"Missing record table:\n{path}"
    )

    schema = pl.read_parquet_schema(
        path
    )

    actual_rows = (
        pl.scan_parquet(path)
        .select(pl.len())
        .collect()
        .item()
    )

    print(
        f"{split:5s}/{source}: "
        f"{actual_rows:,} rows × "
        f"{len(schema)} columns"
    )

    assert actual_rows == expected_rows
    assert len(schema) == 32

    record_inventory[
        f"{split}/{source}"
    ] = {
        "relative_path": str(
            path.relative_to(
                FEATURE_ROOT
            )
        ),
        "rows": int(actual_rows),
        "columns": int(len(schema)),
        "bytes": int(
            path.stat().st_size
        ),
    }

print("✅ All six record tables verified.")

# ------------------------------------------------------------------------------
# 3. VERIFY FEATURE SCHEMA
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. VERIFY FEATURE SCHEMA")
print("-" * 78)

SCHEMA_PATH = (
    FEATURE_ROOT
    / "schema"
    / "feature_schema_v1.json"
)

assert SCHEMA_PATH.exists()

with open(SCHEMA_PATH) as f:
    schema_obj = json.load(f)

registered = None

if isinstance(schema_obj, dict):

    registered = (
        schema_obj.get("columns")
        or schema_obj.get("features")
        or schema_obj.get(
            "registered_columns"
        )
    )

else:

    registered = schema_obj

if registered is not None:

    print(
        "Registered feature columns:",
        len(registered),
    )

    assert len(registered) == 86

print("✅ Schema verified.")

# ------------------------------------------------------------------------------
# 4. FUNCTION — VERIFY SHARDED EMBEDDINGS
# ------------------------------------------------------------------------------

def verify_sharded_embedding_set(
    root: Path,
    expected_rows: int,
    expected_dim: int = 384,
):
    """
    Supports both:
      embeddings_0000.npy + ids_0000.parquet
    and original Cell-64:
      emb_000000000_000100000.npy
      ids_000000000_000100000.parquet
    """

    npy_files = sorted(
        root.glob("*.npy")
    )

    parquet_files = sorted(
        root.glob("*.parquet")
    )

    assert npy_files, (
        f"No NPY files found in {root}"
    )

    assert parquet_files, (
        f"No parquet ID files found in {root}"
    )

    assert len(npy_files) == len(
        parquet_files
    ), (
        f"Embedding/ID shard count mismatch "
        f"in {root}: "
        f"{len(npy_files)} vs "
        f"{len(parquet_files)}"
    )

    verified_rows = 0
    total_emb_bytes = 0
    total_id_bytes = 0

    for emb_path in npy_files:

        # ----------------------------------------------------------------------
        # Parse expected row count from filename where available.
        # Original Cell 64:
        #   emb_000000000_000100000.npy
        # ----------------------------------------------------------------------

        stem = emb_path.stem

        expected_shard_rows = None

        parts = stem.split("_")

        if (
            len(parts) >= 3
            and parts[0] == "emb"
        ):

            try:

                start = int(parts[-2])
                end = int(parts[-1])

                expected_shard_rows = (
                    end - start
                )

            except Exception:
                pass

        arr = np.load(
            emb_path,
            mmap_mode="r",
        )

        assert len(arr.shape) == 2, (
            f"Bad embedding shape: "
            f"{emb_path}: {arr.shape}"
        )

        assert arr.shape[1] == expected_dim, (
            f"Wrong dimension in "
            f"{emb_path}: "
            f"{arr.shape}"
        )

        assert arr.dtype == np.float16, (
            f"Wrong dtype in "
            f"{emb_path}: "
            f"{arr.dtype}"
        )

        actual_shard_rows = int(
            arr.shape[0]
        )

        if expected_shard_rows is not None:

            assert (
                actual_shard_rows
                == expected_shard_rows
            ), (
                f"Filename/data mismatch:\n"
                f"{emb_path}\n"
                f"filename rows="
                f"{expected_shard_rows:,}, "
                f"actual="
                f"{actual_shard_rows:,}"
            )

        verified_rows += (
            actual_shard_rows
        )

        total_emb_bytes += (
            emb_path.stat().st_size
        )

        del arr

    for p in parquet_files:

        rows = (
            pl.scan_parquet(p)
            .select(pl.len())
            .collect()
            .item()
        )

        # ID tables may have both:
        #   row_index
        #   entity_id
        #
        # We only require the expected entity IDs to exist.
        schema = pl.read_parquet_schema(p)

        assert "entity_id" in schema, (
            f"entity_id missing in {p}"
        )

        total_id_rows = rows

        total_id_bytes += (
            p.stat().st_size
        )

        # Keep row totals separately below.

    parquet_rows = sum(
        (
            pl.scan_parquet(p)
            .select(pl.len())
            .collect()
            .item()
        )
        for p in parquet_files
    )

    assert verified_rows == expected_rows, (
        f"Embedding rows mismatch:\n"
        f"{root}\n"
        f"{verified_rows:,} != "
        f"{expected_rows:,}"
    )

    assert parquet_rows == expected_rows, (
        f"ID rows mismatch:\n"
        f"{root}\n"
        f"{parquet_rows:,} != "
        f"{expected_rows:,}"
    )

    return {
        "rows": int(verified_rows),
        "shards": len(npy_files),
        "embedding_files": len(npy_files),
        "id_files": len(parquet_files),
        "embedding_bytes": int(
            total_emb_bytes
        ),
        "id_bytes": int(
            total_id_bytes
        ),
        "dimension": expected_dim,
        "dtype": "float16",
        "normalized_expected": True,
    }

# ------------------------------------------------------------------------------
# 5. VERIFY S1 — ORIGINAL CELL-64 FORMAT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. VERIFY TRAIN / S1 EMBEDDINGS")
print("   ORIGINAL CELL-64 FORMAT")
print("-" * 78)

S1_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s1"
    / "full_record"
)

assert S1_ROOT.exists()

# Read original manifest.
S1_MANIFEST_PATH = (
    S1_ROOT
    / "manifest.json"
)

assert S1_MANIFEST_PATH.exists()

with open(
    S1_MANIFEST_PATH
) as f:

    s1_manifest = json.load(f)

print(
    "Model:",
    s1_manifest.get("model")
)

print(
    "Dimension:",
    s1_manifest.get("dimension")
)

print(
    "Dtype:",
    s1_manifest.get("dtype")
)

print(
    "Rows:",
    s1_manifest.get("rows")
)

print(
    "Shard rows:",
    s1_manifest.get(
        "shard_rows"
    )
)

assert (
    s1_manifest["model"]
    == "intfloat/multilingual-e5-small"
)

assert (
    s1_manifest["dimension"]
    == 384
)

assert (
    s1_manifest["dtype"]
    == "float16"
)

assert (
    s1_manifest["rows"]
    == 2_206_821
)

assert (
    s1_manifest["shard_rows"]
    == 100_000
)

# Manifest itself contains the authoritative shard list.
manifest_shards = (
    s1_manifest["shards"]
)

assert len(manifest_shards) == 23

print(
    f"Manifest shards: "
    f"{len(manifest_shards)}"
)

# Verify every manifest-declared file.
s1_verified_rows = 0
s1_embedding_bytes = 0
s1_id_bytes = 0

for i, shard in enumerate(
    manifest_shards
):

    start = int(
        shard["start"]
    )

    end = int(
        shard["end"]
    )

    expected_rows = (
        end - start
    )

    emb_path = (
        S1_ROOT
        / shard["embedding"]
    )

    id_path = (
        S1_ROOT
        / shard["ids"]
    )

    assert emb_path.exists(), (
        f"Missing S1 embedding shard:\n"
        f"{emb_path}"
    )

    assert id_path.exists(), (
        f"Missing S1 ID shard:\n"
        f"{id_path}"
    )

    arr = np.load(
        emb_path,
        mmap_mode="r",
    )

    assert arr.shape == (
        expected_rows,
        384,
    ), (
        f"Bad S1 shard shape:\n"
        f"{emb_path}\n"
        f"{arr.shape}"
    )

    assert arr.dtype == np.float16

    ids_schema = (
        pl.read_parquet_schema(
            id_path
        )
    )

    assert "entity_id" in ids_schema

    id_rows = (
        pl.scan_parquet(id_path)
        .select(pl.len())
        .collect()
        .item()
    )

    assert id_rows == expected_rows, (
        f"ID row mismatch:\n"
        f"{id_path}"
    )

    s1_verified_rows += (
        expected_rows
    )

    s1_embedding_bytes += (
        emb_path.stat().st_size
    )

    s1_id_bytes += (
        id_path.stat().st_size
    )

    del arr

    print(
        f"[S1 {i+1:02d}/23] "
        f"{start:,}:{end:,} "
        f"rows={expected_rows:,} ✅"
    )

assert (
    s1_verified_rows
    == 2_206_821
)

print(
    "\n✅ S1 COMPLETE — "
    f"{s1_verified_rows:,} rows verified."
)

# ------------------------------------------------------------------------------
# 6. CREATE S1 COMPLETE MARKER NOW
# ------------------------------------------------------------------------------

S1_COMPLETE_MARKER = (
    S1_ROOT
    / "COMPLETE"
)

S1_COMPLETE_MARKER.write_text(
    "CELL 64 COMPLETE\n"
    "TRAIN S1 EMBEDDINGS COMPLETE\n"
    "Verified by MASTER CHECKPOINT AFTER CELL 66.\n"
    "Rows=2206821\n"
    "Shards=23\n"
    "Dimension=384\n"
    "Dtype=float16\n"
    "Model=intfloat/multilingual-e5-small\n"
)

print(
    "✅ S1 COMPLETE marker written:"
)

print(
    S1_COMPLETE_MARKER
)

# ------------------------------------------------------------------------------
# 7. VERIFY S2
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. VERIFY TRAIN / S2 EMBEDDINGS")
print("-" * 78)

S2_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s2"
    / "full_record"
)

assert (
    S2_ROOT
    / "manifest.json"
).exists()

assert (
    S2_ROOT
    / "COMPLETE"
).exists()

s2_info = (
    verify_sharded_embedding_set(
        S2_ROOT,
        5_034_616,
        384,
    )
)

with open(
    S2_ROOT / "manifest.json"
) as f:

    s2_manifest = json.load(f)

assert s2_manifest["status"] == "COMPLETE"
assert s2_manifest["rows"] == 5_034_616
assert s2_manifest["shards"] == 51
assert s2_manifest["dtype"] == "float16"
assert s2_manifest["normalized"] is True

print(
    f"✅ S2: "
    f"{s2_info['rows']:,} rows / "
    f"{s2_info['shards']} shards"
)

# ------------------------------------------------------------------------------
# 8. VERIFY S3
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. VERIFY TRAIN / S3 EMBEDDINGS")
print("-" * 78)

S3_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s3"
    / "full_record"
)

assert (
    S3_ROOT
    / "manifest.json"
).exists()

assert (
    S3_ROOT
    / "COMPLETE"
).exists()

s3_info = (
    verify_sharded_embedding_set(
        S3_ROOT,
        5_285_603,
        384,
    )
)

with open(
    S3_ROOT / "manifest.json"
) as f:

    s3_manifest = json.load(f)

assert s3_manifest["status"] == "COMPLETE"
assert s3_manifest["rows"] == 5_285_603
assert s3_manifest["shards"] == 53
assert s3_manifest["dtype"] == "float16"
assert s3_manifest["normalized"] is True

print(
    f"✅ S3: "
    f"{s3_info['rows']:,} rows / "
    f"{s3_info['shards']} shards"
)

# ------------------------------------------------------------------------------
# 9. BUILD MASTER INVENTORY
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. BUILD MASTER CHECKPOINT INVENTORY")
print("-" * 78)

embedding_inventory = {

    "s1": {
        "layout":
            "original_cell64_manifest_shards",

        "rows":
            s1_verified_rows,

        "shards":
            23,

        "dimension":
            384,

        "dtype":
            "float16",

        "embedding_bytes":
            s1_embedding_bytes,

        "id_bytes":
            s1_id_bytes,

        "complete_marker":
            True,
    },

    "s2": s2_info,

    "s3": s3_info,
}

TOTAL_EMBED_BYTES = sum(
    x["embedding_bytes"]
    for x in embedding_inventory.values()
)

TOTAL_ID_BYTES = sum(
    x["id_bytes"]
    for x in embedding_inventory.values()
)

# ------------------------------------------------------------------------------
# 10. SAVE MASTER CHECKPOINT
# ------------------------------------------------------------------------------

CHECKPOINT_ROOT = Path(
    "/kaggle/working/AMLC2026/"
    "checkpoint_after_cell66_ALL_TRAIN_EMBEDDINGS_20260926"
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_PATH = (
    CHECKPOINT_ROOT
    / "CHECKPOINT_METADATA.json"
)

INVENTORY_PATH = (
    CHECKPOINT_ROOT
    / "FEATURE_LAKE_INVENTORY.json"
)

MASTER_MARKER = (
    CHECKPOINT_ROOT
    / "CELL66_ALL_TRAIN_EMBEDDINGS_COMPLETE"
)

inventory = {

    "checkpoint":
        "AMLC2026_AFTER_CELL66",

    "state":
        "ALL_TRAIN_EMBEDDINGS_COMPLETE",

    "feature_root":
        str(FEATURE_ROOT),

    "cell61":
        "COMPLETE",

    "cell62":
        "COMPLETE",

    "cell63":
        "COMPLETE",

    "cell64":
        "COMPLETE",

    "cell65":
        "COMPLETE",

    "cell66":
        "COMPLETE",

    "train_embeddings":
        embedding_inventory,

    "records":
        record_inventory,

    "next_stage":
        "PAIR_LEVEL_SEMANTIC_FEATURES",

    "zip_created":
        False,

    "feature_lake_is_payload":
        True,

    "created_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

with open(
    INVENTORY_PATH,
    "w",
) as f:

    json.dump(
        inventory,
        f,
        indent=2,
    )

metadata = {

    "checkpoint_name":
        "AMLC2026_AFTER_CELL66",

    "state":
        "ALL_TRAIN_EMBEDDINGS_COMPLETE",

    "feature_root":
        str(FEATURE_ROOT),

    "train_s1_rows":
        2_206_821,

    "train_s2_rows":
        5_034_616,

    "train_s3_rows":
        5_285_603,

    "train_s1_verified":
        True,

    "train_s2_verified":
        True,

    "train_s3_verified":
        True,

    "s1_marker_created":
        True,

    "s1_layout":
        "Cell64_original_manifest_shards",

    "s2_layout":
        "Cell65_sharded_npy",

    "s3_layout":
        "Cell66_sharded_npy",

    "total_embedding_bytes":
        TOTAL_EMBED_BYTES,

    "total_id_bytes":
        TOTAL_ID_BYTES,

    "next_stage":
        "PAIR_LEVEL_SEMANTIC_FEATURES",

    "zip_created":
        False,

    "created_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

with open(
    METADATA_PATH,
    "w",
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
    )

MASTER_MARKER.write_text(
    "AMLC 2026 MASTER CHECKPOINT\n"
    "CELL 61 COMPLETE\n"
    "CELL 62 COMPLETE\n"
    "CELL 63 COMPLETE\n"
    "CELL 64 COMPLETE\n"
    "CELL 65 COMPLETE\n"
    "CELL 66 COMPLETE\n"
    "TRAIN S1 EMBEDDINGS COMPLETE\n"
    "TRAIN S2 EMBEDDINGS COMPLETE\n"
    "TRAIN S3 EMBEDDINGS COMPLETE\n"
    "NEXT=PAIR_LEVEL_SEMANTIC_FEATURES\n"
)

# ------------------------------------------------------------------------------
# 11. FINAL CLEANUP
# ------------------------------------------------------------------------------

gc.collect()

try:

    import torch

    if torch.cuda.is_available():

        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

except Exception:
    pass

elapsed = (
    time.time() - T0
) / 60

print("\n" + "=" * 78)
print("✅✅✅ MASTER CHECKPOINT AFTER CELL 66 COMPLETE ✅✅✅")
print("=" * 78)

print(f"""
FEATURE ROOT:
  {FEATURE_ROOT}

S1:
  2,206,821 rows ✅
  23 shards ✅
  Original Cell-64 format verified ✅
  COMPLETE marker created ✅

S2:
  5,034,616 rows ✅
  51 shards ✅
  COMPLETE marker verified ✅

S3:
  5,285,603 rows ✅
  53 shards ✅
  COMPLETE marker verified ✅

TOTAL TRAIN EMBEDDING STORAGE:
  {TOTAL_EMBED_BYTES / (1024**3):.2f} GB

TOTAL ID STORAGE:
  {TOTAL_ID_BYTES / (1024**3):.2f} GB

CHECKPOINT:
  {CHECKPOINT_ROOT}

METADATA:
  {METADATA_PATH}

INVENTORY:
  {INVENTORY_PATH}

ZIP:
  ❌ NONE

NEXT:
  PAIR-LEVEL SEMANTIC FEATURES

✅ NO EMBEDDINGS WERE RECOMPUTED.
✅ NO FEATURE-LAKE DATA WAS COPIED.
✅ ORIGINAL S1 FORMAT WAS PRESERVED.
✅ ALL TRAIN EMBEDDING DATA IS VERIFIED.
""")

print(
    f"Checkpoint runtime: "
    f"{elapsed:.2f} min"
)

AMLC 2026 — MASTER CHECKPOINT AFTER CELL 66
FINAL FIX — ORIGINAL CELL-64 S1 FORMAT

------------------------------------------------------------------------------
0. RAM CLEANUP
------------------------------------------------------------------------------
RAM: 24.55 / 31.35 GB (78.9%)

------------------------------------------------------------------------------
1. LOCATE FEATURE LAKE
------------------------------------------------------------------------------
Candidates:
  - /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

✅ FEATURE_ROOT = /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

------------------------------------------------------------------------------
2. VERIFY RECORD LAKE
------------------------------------------------------------------------------
train/s1: 2,206,821 rows × 32 columns
train/s2: 5,034,616 rows × 32 columns
train/s3: 5,285,603 rows × 32 columns
test /s1: 1,732,544 rows × 32 columns
test /s2: 4,887,273 rows × 32 columns
test /s3: 5,082,316 rows × 32 co